In [1]:
# Clone your GitHub repo (you’ll be prompted to authorize if it's private)
!git clone https://github.com/colterwood/LHL-final-final-project.git

Cloning into 'LHL-final-final-project'...
remote: Enumerating objects: 394, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (158/158), done.
remote: Total 394 (delta 85), reused 50 (delta 12), pack-reused 224 (from 1)
Receiving objects: 100% (394/394), 6.06 MiB | 2.94 MiB/s, done.
Resolving deltas: 100% (200/200), done.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, ParameterGrid
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from google.colab import files
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
import json
import os
from sklearn.model_selection import cross_val_score

In [ ]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/2024_persona_and_team_data.csv")


# preview
df.head()

,team,opp,month,day,team_score,opp_score,elite_scorer,efficient_scorer,volume_shooter,three_point_specialist,...,opp_shooting_FG% by Distance_2P,opp_shooting_FG% by Distance_0-3,opp_shooting_FG% by Distance_3-10,opp_shooting_FG% by Distance_10-16,opp_shooting_FG% by Distance_16-3P,opp_shooting_FG% by Distance_3P,opp_shooting_% of FG Ast'd_2P,opp_shooting_% of FG Ast'd_3P,opp_shooting_Corner_%3PA,opp_shooting_Corner_3P%
0,ATL,LAS,5,15,92,81,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
1,ATL,PHO,5,18,85,88,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
2,ATL,DAL,5,21,83,78,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
3,ATL,MIN,5,26,79,92,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357
4,ATL,WAS,5,29,73,67,0,0,1,1,...,0.473,0.617,0.4,0.358,0.397,0.344,0.628,0.874,0.215,0.357


In [ ]:
df.shape

(480, 218)

In [ ]:
# models to train: 12 teams + league-wide
teams = df["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df.copy()
    else:
        df_team = df[df["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics (just MAE + R²)
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# preview
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
47,team_per_game_PTS,0.056100
35,team_per_game_2P%,0.028499
27,team_per_game_FG,0.026299
137,opp_per_game_2P,0.018609
0,elite_scorer,0.018514
...,...,...
201,opp_shooting_% of FGA by Distance_3P,0.000000
202,opp_shooting_FG% by Distance_2P,0.000000
207,opp_shooting_FG% by Distance_3P,0.000000
204,opp_shooting_FG% by Distance_3-10,0.000000


In [ ]:
# drop features from df with team_score importance < 0.001
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.001]["Feature"].tolist()
filtered_df_v1 = df.drop(columns=low_importance_cols, errors="ignore")

In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v1["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v1.copy()
    else:
        df_team = filtered_df_v1[filtered_df_v1["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# preview
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
45,team_per_game_PTS,0.056100
33,team_per_game_2P%,0.028499
25,team_per_game_FG,0.026299
82,opp_per_game_2P,0.018609
0,elite_scorer,0.018514
...,...,...
40,team_per_game_AST,0.003883
31,team_per_game_2P,0.003736
29,team_per_game_3PA,0.003439
90,opp_per_game_PF,0.003209


In [ ]:
# drop features from df with team_score importance < 0.005
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.005]["Feature"].tolist()
filtered_df_v2 = df.drop(columns=low_importance_cols, errors="ignore")

In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v2["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v2.copy()
    else:
        df_team = filtered_df_v2[filtered_df_v2["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.314095,27.683189,10.343696,8.683189,-1.132368
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400055,25.781090,11.478148,11.791771,-1.399221
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.417870,21.607391,8.215759,6.475475,-0.650818
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.133636,6.471183,6.250748,0.216070
9,PHO,1.229004,16.355804,8.403378,9.037724,0.400211


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
41,team_per_game_PTS,0.058853
127,opp_per_game_2P,0.036956
25,team_per_game_FG,0.028111
30,team_per_game_2P%,0.026514
115,team_shooting_FG% by Distance_10-16,0.025126
...,...,...
183,opp_shooting_FG%,0.000000
190,opp_shooting_FG% by Distance_2P,0.000000
189,opp_shooting_% of FGA by Distance_3P,0.000000
184,opp_shooting_Dist.,0.000000


In [ ]:
# drop features from df with team_score importance < 0.01
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.01]["Feature"].tolist()
filtered_df_v3 = df.drop(columns=low_importance_cols, errors="ignore")

In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v3["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v3.copy()
    else:
        df_team = filtered_df_v3[filtered_df_v3["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.499786,30.174622,8.883224,5.837204,-0.876047
1,CHI,1.483910,21.510834,10.403218,11.016090,-0.479654
2,CON,2.471138,39.884995,11.551542,7.080780,-1.840674
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.224800,23.576385,8.425079,7.000000,-0.744305
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.660637,16.956604,10.290885,11.308620,0.183954


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,team_per_game_PTS,0.057869
20,team_W,0.050538
15,team_per_game_BLK,0.035029
10,team_per_game_2P%,0.033771
11,team_per_game_FT%,0.032411
39,opp_per_poss_FGA,0.031137
41,opp_shooting_% of FGA by Distance_10-16,0.030505
25,team_shooting_% of FGA by Distance_3-10,0.030012
0,elite_scorer,0.028974
22,team_TOV%.1,0.028882


In [ ]:
# drop 'opp_totals_2P' only
filtered_df_v4 = filtered_df_v3.drop(columns=["opp_totals_2P"], errors="ignore")

In [ ]:
filtered_df_v4.shape

(480, 48)

In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v4["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v4.copy()
    else:
        df_team = filtered_df_v4[filtered_df_v4["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.499786,30.174622,8.883224,5.837204,-0.876047
1,CHI,1.483910,21.510834,10.403218,11.016090,-0.479654
2,CON,2.471138,39.884995,11.551542,7.080780,-1.840674
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.224800,23.576385,8.425079,7.000000,-0.744305
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.660637,16.956604,10.290885,11.308620,0.183954


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,team_per_game_PTS,0.057869
20,team_W,0.050538
15,team_per_game_BLK,0.035029
10,team_per_game_2P%,0.033771
11,team_per_game_FT%,0.032411
38,opp_per_poss_FGA,0.031137
40,opp_shooting_% of FGA by Distance_10-16,0.030505
25,team_shooting_% of FGA by Distance_3-10,0.030012
0,elite_scorer,0.028974
22,team_TOV%.1,0.028882


In [ ]:
# drop 'opp_totals_2P' only
filtered_df_v5 = filtered_df_v4.drop(columns=["three_point_specialist"], errors="ignore")

In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v5["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v5.copy()
    else:
        df_team = filtered_df_v5[filtered_df_v5["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.112991,28.431786,8.927518,5.159397,-0.740086
1,CHI,0.930618,25.069382,9.913876,9.000000,-0.398478
2,CON,1.990623,39.915024,11.439304,6.509377,-1.823733
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.073547,24.424126,8.464477,7.000000,-0.792963
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.335999,16.425278,9.841609,10.624851,0.233447


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
17,team_per_game_PTS,0.053061
19,team_W,0.044506
39,opp_shooting_% of FGA by Distance_10-16,0.038597
14,team_per_game_BLK,0.035138
37,opp_per_poss_FGA,0.031932
9,team_per_game_2P%,0.031847
40,opp_shooting_FG% by Distance_0-3,0.031678
20,team_ORB%,0.030974
24,team_shooting_% of FGA by Distance_3-10,0.030469
33,opp_per_game_2PA,0.029217


In [ ]:
# drop "team_W" from v4
filtered_df_v6 = filtered_df_v4.drop(columns=["team_W"], errors="ignore")

In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v6["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v6.copy()
    else:
        df_team = filtered_df_v6[filtered_df_v6["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.499786,30.174622,8.883224,5.837204,-0.876047
1,CHI,1.483910,21.510834,10.403218,11.016090,-0.479654
2,CON,2.471138,39.884995,11.551542,7.080780,-1.840674
3,DAL,2.604134,20.604469,9.005244,7.604301,-1.028053
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.317802,20.682198,9.347076,8.000000,-0.186094
6,LVA,1.224800,23.576385,8.425079,7.000000,-0.744305
7,MIN,1.260544,24.861389,11.107976,9.500000,0.169617
8,NYL,5.981270,19.981270,9.692212,7.981270,-0.293668
9,PHO,2.660637,16.956604,10.290885,11.308620,0.183954


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,team_per_game_PTS,0.053040
15,team_per_game_BLK,0.036429
10,team_per_game_2P%,0.036074
37,opp_per_poss_FGA,0.036018
39,opp_shooting_% of FGA by Distance_10-16,0.034331
20,team_ORB%,0.033452
24,team_shooting_% of FGA by Distance_3-10,0.032021
0,elite_scorer,0.031700
33,opp_per_game_2PA,0.031511
11,team_per_game_FT%,0.031503


In [ ]:
# drop team_W and team_L from df, bring the rest back
filtered_df_v7 = df.drop(columns=["team_W", "team_L"], errors="ignore")

In [ ]:
for col in filtered_df_v7:
  print(col)

team
opp
month
day
team_score
opp_score
elite_scorer
efficient_scorer
volume_shooter
three_point_specialist
slasher
free_throw_generator
and_one_machine
playmaker
offensive_hub
turnover_prone
floor_general
plus_minus_driver
self_creator
rim_protector
steal_artist
defensive_anchor
glass_cleaner
offensive_rebounder
defensive_rebounder
midrange_sniper
corner_3_specialist
catch_and_shoot
stretch_big
heave_chucker
all_around_star
impact_bench
fast_break_threat
team_per_game_FG
team_per_game_FGA
team_per_game_FG%
team_per_game_3P
team_per_game_3PA
team_per_game_3P%
team_per_game_2P
team_per_game_2PA
team_per_game_2P%
team_per_game_FT
team_per_game_FTA
team_per_game_FT%
team_per_game_ORB
team_per_game_DRB
team_per_game_TRB
team_per_game_AST
team_per_game_STL
team_per_game_BLK
team_per_game_TOV
team_per_game_PF
team_per_game_PTS
team_totals_FG
team_totals_FGA
team_totals_FG%
team_totals_3P
team_totals_3PA
team_totals_3P%
team_totals_2P
team_totals_2PA
team_totals_2P%
team_totals_FT
team_totals

In [ ]:
# drop non-feature columns
X = filtered_df_v7.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = filtered_df_v7["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# train baseline model
model = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=1,
    min_child_weight=1,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)

print(f"Trained model on {X_train.shape[0]} rows and {X_train.shape[1]} features.\n")

Trained model on 360 rows and 210 features.



In [ ]:
# get feature names
feature_names = X_train.columns.tolist()

# extract importances
importances_team = model.feature_importances_

# build importance DataFrame
df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
})

# show 0-importance features
zero_importance_cols = df_importances[df_importances["Importance_team_score"] == 0]["Feature"].tolist()
print(f"Found {len(zero_importance_cols)} features with 0 importance:\n{zero_importance_cols}\n")

# drop them to create v7
filtered_df_v7 = filtered_df_v7.drop(columns=zero_importance_cols, errors="ignore")

Found 98 features with 0 importance:
['floor_general', 'stretch_big', 'team_totals_FG', 'team_totals_FGA', 'team_totals_FG%', 'team_totals_3P', 'team_totals_3PA', 'team_totals_3P%', 'team_totals_2P', 'team_totals_2PA', 'team_totals_2P%', 'team_totals_FT', 'team_totals_FTA', 'team_totals_FT%', 'team_totals_DRB', 'team_totals_TRB', 'team_totals_BLK', 'team_totals_TOV', 'team_totals_PF', 'team_totals_PTS', 'team_PL', 'team_MOV', 'team_SOS', 'team_SRS', 'team_NRtg', 'team_FTr', 'team_3PAr', 'team_TS%', 'team_eFG%', 'team_TOV%', 'team_per_poss_FG%', 'team_per_poss_3P', 'team_per_poss_3PA', 'team_per_poss_3P%', 'team_per_poss_2P', 'team_per_poss_2PA', 'team_per_poss_2P%', 'team_per_poss_FT', 'team_per_poss_FTA', 'team_per_poss_FT%', 'team_per_poss_ORB', 'team_per_poss_TRB', 'team_per_poss_AST', 'team_per_poss_STL', 'team_per_poss_BLK', 'team_per_poss_TOV', 'team_per_poss_PTS', 'team_shooting_FG%', 'team_shooting_Dist.', 'team_shooting_% of FGA by Distance_2P', 'team_shooting_% of FGA by Dist

In [ ]:
# drop non-feature columns
X = filtered_df_v7.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.9
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

all_around_star ⬌ defensive_anchor | correlation = 0.924
team_per_game_3PA ⬌ team_per_game_3P | correlation = 0.965
team_per_game_2PA ⬌ team_per_game_3P | correlation = 0.926
team_per_game_2PA ⬌ team_per_game_3PA | correlation = 0.945
team_per_game_2PA ⬌ team_per_game_2P | correlation = 0.900
team_per_game_FTA ⬌ team_per_game_FT | correlation = 0.915
team_totals_ORB ⬌ team_per_game_ORB | correlation = 1.000
team_totals_AST ⬌ team_per_game_AST | correlation = 1.000
team_totals_STL ⬌ team_per_game_STL | correlation = 1.000
team_PW ⬌ all_around_star | correlation = 0.905
team_ORtg ⬌ team_per_game_PTS | correlation = 0.914
team_ORB% ⬌ team_per_game_ORB | correlation = 0.983
team_ORB% ⬌ team_totals_ORB | correlation = 0.983
team_FT/FGA ⬌ team_per_game_FT | correlation = 0.954
team_FT/FGA ⬌ team_per_game_FTA | correlation = 0.919
team_eFG%.1 ⬌ team_DRtg | correlation = 0.916
team_per_poss_FG ⬌ team_per_game_FG | correlation = 0.932
team_per_poss_DRB ⬌ team_per_game_DRB | correlation = 0.953


In [ ]:
# Trim multicollinearity
drop_cols = [
    "team_per_game_3P",
    "team_per_game_3PA",
    "team_per_game_2P",
    "team_per_game_FT",
    "team_totals_ORB",
    "team_totals_AST",
    "team_totals_STL",
    "team_per_game_PTS",
    "team_ORB%",
    "team_per_game_FTA",
    "team_eFG%.1",
    "team_per_game_FG",
    "team_per_game_DRB",
    "team_per_game_PF",
    "opp_per_game_FG",
    "opp_per_game_FG%",  # ← You meant this when referencing team_eFG%.1 links
    "opp_per_game_FT",
    "team_DRB%",
    "team_TOV%.1",
    "opp_totals_2P",
    "opp_totals_TOV",
    "opp_per_poss_FG",
    "opp_per_game_DRB",
    "opp_per_game_TRB",
    "opp_per_game_PF",
    "opp_shooting_% of FGA by Distance_2P"
]

filtered_df_v7 = filtered_df_v7.drop(columns=drop_cols, errors="ignore")


Dropped 26 columns. filtered_df_v8 now has 118 columns.



In [ ]:
print(f"Dropped {len(drop_cols)} columns. filtered_df_v7 now has {filtered_df_v7.shape[1]} columns.\n")

Dropped 26 columns. filtered_df_v7 now has 92 columns.



In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v7["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v7.copy()
    else:
        df_team = filtered_df_v7[filtered_df_v7["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
44,team_per_poss_FG,0.076448
29,team_per_game_2P%,0.043564
0,elite_scorer,0.022937
46,team_per_poss_DRB,0.020471
54,team_shooting_FG% by Distance_10-16,0.019813
...,...,...
75,opp_shooting_% of FGA by Distance_0-3,0.005720
79,opp_shooting_FG% by Distance_0-3,0.005678
73,opp_per_poss_TRB,0.005668
7,playmaker,0.005534


In [ ]:
# drop team_W and team_L from df, bring the rest back
filtered_df_v8 = df.drop(columns=["team_W", "team_L"], errors="ignore")

In [ ]:
# drop non-feature columns
X = filtered_df_v8.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = filtered_df_v8["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# train baseline model
model = XGBRegressor(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=1,
    min_child_weight=1,
    random_state=42,
    verbosity=0
)
model.fit(X_train, y_train)

print(f"Trained model on {X_train.shape[0]} rows and {X_train.shape[1]} features.\n")

Trained model on 360 rows and 210 features.



In [ ]:
# get feature names
feature_names = X_train.columns.tolist()

# extract importances
importances_team = model.feature_importances_

# build importance DataFrame
df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
})

# show 0-importance features
zero_importance_cols = df_importances[df_importances["Importance_team_score"] == 0]["Feature"].tolist()
print(f"Found {len(zero_importance_cols)} features with 0 importance:\n{zero_importance_cols}\n")

# drop them to create v7
filtered_df_v8 = filtered_df_v8.drop(columns=zero_importance_cols, errors="ignore")

Found 98 features with 0 importance:
['floor_general', 'stretch_big', 'team_totals_FG', 'team_totals_FGA', 'team_totals_FG%', 'team_totals_3P', 'team_totals_3PA', 'team_totals_3P%', 'team_totals_2P', 'team_totals_2PA', 'team_totals_2P%', 'team_totals_FT', 'team_totals_FTA', 'team_totals_FT%', 'team_totals_DRB', 'team_totals_TRB', 'team_totals_BLK', 'team_totals_TOV', 'team_totals_PF', 'team_totals_PTS', 'team_PL', 'team_MOV', 'team_SOS', 'team_SRS', 'team_NRtg', 'team_FTr', 'team_3PAr', 'team_TS%', 'team_eFG%', 'team_TOV%', 'team_per_poss_FG%', 'team_per_poss_3P', 'team_per_poss_3PA', 'team_per_poss_3P%', 'team_per_poss_2P', 'team_per_poss_2PA', 'team_per_poss_2P%', 'team_per_poss_FT', 'team_per_poss_FTA', 'team_per_poss_FT%', 'team_per_poss_ORB', 'team_per_poss_TRB', 'team_per_poss_AST', 'team_per_poss_STL', 'team_per_poss_BLK', 'team_per_poss_TOV', 'team_per_poss_PTS', 'team_shooting_FG%', 'team_shooting_Dist.', 'team_shooting_% of FGA by Distance_2P', 'team_shooting_% of FGA by Dist

In [ ]:
# drop non-feature columns
X = filtered_df_v8.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.99
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_totals_ORB ⬌ team_per_game_ORB | correlation = 1.000
team_totals_AST ⬌ team_per_game_AST | correlation = 1.000
team_totals_STL ⬌ team_per_game_STL | correlation = 1.000
opp_totals_2P ⬌ opp_per_game_2P | correlation = 1.000
opp_totals_TOV ⬌ opp_per_game_TOV | correlation = 1.000


In [ ]:
# columns with perfect 1.000 correlation to other features
drop_cols_v8 = [
    "team_totals_ORB",
    "team_totals_AST",
    "team_totals_STL",
    "opp_totals_2P",
    "opp_totals_TOV"
]

# drop from filtered_df_v8
filtered_df_v8 = filtered_df_v8.drop(columns=drop_cols_v8, errors="ignore")
print(f"Dropped {len(drop_cols_v8)} columns from filtered_df_v8 to update filtered_df_v8.")


Dropped 5 columns from filtered_df_v8 to update filtered_df_v8.


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v8["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v8.copy()
    else:
        df_team = filtered_df_v8[filtered_df_v8["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
45,team_per_game_PTS,0.058535
33,team_per_game_2P%,0.036860
25,team_per_game_FG,0.028228
42,team_per_game_BLK,0.020093
79,opp_per_game_2P,0.019370
...,...,...
48,team_ORtg,0.000000
74,opp_per_game_FGA,0.000000
73,opp_per_game_FG,0.000000
61,team_shooting_% of FGA by Distance_0-3,0.000000


In [ ]:
# drop non-feature columns
X = filtered_df_v8.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.95
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_3PA ⬌ team_per_game_3P | correlation = 0.965
team_ORB% ⬌ team_per_game_ORB | correlation = 0.983
team_FT/FGA ⬌ team_per_game_FT | correlation = 0.954
team_per_poss_DRB ⬌ team_per_game_DRB | correlation = 0.953
team_per_poss_PF ⬌ team_per_game_PF | correlation = 0.974
opp_per_game_FG% ⬌ team_eFG%.1 | correlation = 0.971
opp_per_game_FT ⬌ team_FT/FGA.1 | correlation = 0.955
opp_per_game_TOV ⬌ team_TOV%.1 | correlation = 0.974
opp_per_poss_FG ⬌ opp_per_game_FG | correlation = 0.974


In [ ]:
drop_cols_v9 = [
    "team_per_game_ORB",
    "team_per_game_PF",
    "opp_per_game_TOV",
    "opp_per_game_FG",
    "team_per_game_3P",
    "team_per_game_DRB",
    "opp_per_game_FT",
    "opp_per_game_FG%"
]

filtered_df_v9 = filtered_df_v8.drop(columns=drop_cols_v9, errors="ignore")
print(f"Dropped {len(drop_cols_v9)} columns from filtered_df_v8 to create filtered_df_v9.")

Dropped 8 columns from filtered_df_v8 to create filtered_df_v9.


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v9["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v9.copy()
    else:
        df_team = filtered_df_v9[filtered_df_v9["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
41,team_per_game_PTS,0.036841
32,team_per_game_2P%,0.033268
25,team_per_game_FG,0.027385
55,team_per_poss_DRB,0.021553
73,opp_per_game_2P,0.021196
...,...,...
28,team_per_game_3PA,0.004981
80,opp_per_game_PF,0.003944
44,team_ORtg,0.003358
30,team_per_game_2P,0.003019


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v7["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v7.copy()
    else:
        df_team = filtered_df_v7[filtered_df_v7["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.321198,29.136726,10.720178,8.663055,-1.233348
1,CHI,1.412910,25.685169,11.507663,12.747425,-0.895352
2,CON,0.400223,25.781212,11.478449,11.792282,-1.399245
3,DAL,2.919281,38.779213,10.039078,5.380623,-2.569264
4,IND,1.068237,17.915863,7.463308,6.076187,-0.257784
5,LAS,0.113747,21.071144,9.051656,7.913082,-0.175874
6,LVA,0.433884,23.135651,8.369606,6.482758,-0.747650
7,MIN,3.942062,19.752930,10.880116,11.007225,0.280743
8,NYL,1.300591,20.138535,6.471673,6.250748,0.215845
9,PHO,1.186211,16.349876,8.493648,9.020622,0.389254


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
44,team_per_poss_FG,0.076448
29,team_per_game_2P%,0.043564
0,elite_scorer,0.022937
46,team_per_poss_DRB,0.020471
54,team_shooting_FG% by Distance_10-16,0.019813
...,...,...
75,opp_shooting_% of FGA by Distance_0-3,0.005720
79,opp_shooting_FG% by Distance_0-3,0.005678
73,opp_per_poss_TRB,0.005668
7,playmaker,0.005534


In [ ]:
low_importance_cols = df_importances[df_importances["Importance_team_score"] < 0.01]["Feature"].tolist()
print(f"Dropping {len(low_importance_cols)} features with Importance < 0.01 from filtered_df_v7.\n")

Dropping 43 features with Importance < 0.01 from filtered_df_v7.



In [ ]:
filtered_df_v10 = filtered_df_v7.drop(columns=low_importance_cols, errors="ignore")
print(f"filtered_df_v10 created with {filtered_df_v10.shape[1]} columns.")

filtered_df_v10 created with 49 columns.


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v10["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v10.copy()
    else:
        df_team = filtered_df_v10[filtered_df_v10["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.609108,29.390892,9.112714,5.390892,-0.865113
1,CHI,1.488823,19.511177,10.402235,11.011177,-0.459888
2,CON,2.643883,39.881752,11.736557,8.000000,-1.936557
3,DAL,2.616806,20.670052,8.952136,6.209988,-1.078178
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.490883,20.509117,9.361225,8.000000,-0.188345
6,LVA,0.991287,26.916931,8.689079,7.000000,-0.970116
7,MIN,1.138664,25.420624,11.406569,10.114536,0.136061
8,NYL,0.863503,16.549164,4.902157,3.506901,0.518265
9,PHO,2.320305,16.400101,9.843137,10.624023,0.233446


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
20,team_per_poss_FG,0.096284
38,opp_per_poss_FGA,0.039444
10,team_per_game_2P%,0.034924
0,elite_scorer,0.032104
18,team_ORtg,0.026793
15,team_per_game_BLK,0.026729
22,team_shooting_% of FGA by Distance_3-10,0.026679
11,team_per_game_FT%,0.025068
7,team_per_game_FGA,0.024895
23,team_shooting_% of FGA by Distance_16-3P,0.024879


In [ ]:
# drop non-feature columns
X = filtered_df_v10.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.90
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_PW ⬌ all_around_star | correlation = 0.905


In [ ]:
# drop team_PW from filtered_df_v10 to create filtered_df_v11
filtered_df_v11 = filtered_df_v10.drop(columns=["team_PW"], errors="ignore")
print("Dropped 'team_PW' from filtered_df_v10 to create filtered_df_v11.")

Dropped 'team_PW' from filtered_df_v10 to create filtered_df_v11.


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v11["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v11.copy()
    else:
        df_team = filtered_df_v11[filtered_df_v11["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.609108,29.390892,9.112714,5.390892,-0.865113
1,CHI,1.488823,19.511177,10.402235,11.011177,-0.459888
2,CON,2.643883,39.881752,11.736557,8.000000,-1.936557
3,DAL,2.616806,20.670052,8.952136,6.209988,-1.078178
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.490883,20.509117,9.361225,8.000000,-0.188345
6,LVA,0.991287,26.916931,8.689079,7.000000,-0.970116
7,MIN,1.138664,25.420624,11.406569,10.114536,0.136061
8,NYL,0.863503,16.549164,4.902157,3.506901,0.518265
9,PHO,2.320305,16.400101,9.843137,10.624023,0.233446


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
19,team_per_poss_FG,0.098074
37,opp_per_poss_FGA,0.040177
10,team_per_game_2P%,0.035573
0,elite_scorer,0.032701
17,team_ORtg,0.027291
15,team_per_game_BLK,0.027226
21,team_shooting_% of FGA by Distance_3-10,0.027175
11,team_per_game_FT%,0.025534
7,team_per_game_FGA,0.025358
22,team_shooting_% of FGA by Distance_16-3P,0.025341


In [ ]:
# drop non-feature columns
X = filtered_df_v11.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.85
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_2P% ⬌ team_per_game_FG% | correlation = 0.858
team_per_game_BLK ⬌ team_per_game_FT% | correlation = 0.873
team_ORtg ⬌ team_per_game_FG% | correlation = 0.864
team_ORtg ⬌ team_per_game_2P% | correlation = 0.868
team_shooting_FG% by Distance_3-10 ⬌ team_per_game_2P% | correlation = 0.869
opp_per_game_3PA ⬌ opp_per_game_3P | correlation = 0.896
opp_per_poss_FGA ⬌ opp_per_game_FGA | correlation = 0.886


In [ ]:
# drop selected columns from filtered_df_v11
drop_cols_v12 = [
    "team_per_game_FG%",
    "team_per_game_2P%",
    "opp_per_game_3P",
    "opp_per_game_FGA"
]

filtered_df_v12 = filtered_df_v11.drop(columns=drop_cols_v12, errors="ignore")
print(f"Dropped {len(drop_cols_v12)} columns from filtered_df_v11 to create filtered_df_v12.")

Dropped 4 columns from filtered_df_v11 to create filtered_df_v12.


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v12["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v12.copy()
    else:
        df_team = filtered_df_v12[filtered_df_v12["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.609108,29.390892,9.112714,5.390892,-0.865113
1,CHI,1.488823,19.511177,10.402235,11.011177,-0.459888
2,CON,2.643883,39.881752,11.736557,8.000000,-1.936557
3,DAL,2.616806,20.670052,8.952136,6.209988,-1.078178
4,IND,0.620049,17.931244,8.251981,7.379951,-0.491327
5,LAS,1.490883,20.509117,9.361225,8.000000,-0.188345
6,LVA,0.991287,26.916931,8.689079,7.000000,-0.970116
7,MIN,1.138664,25.420624,11.406569,10.114536,0.136061
8,NYL,0.863503,16.549164,4.902157,3.506901,0.518265
9,PHO,2.320305,16.400101,9.843137,10.624023,0.233446


In [ ]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
17,team_per_poss_FG,0.077675
33,opp_per_poss_FGA,0.044730
0,elite_scorer,0.039040
13,team_per_game_BLK,0.038586
36,opp_shooting_% of FG Ast'd_3P,0.033695
15,team_ORtg,0.033198
19,team_shooting_% of FGA by Distance_3-10,0.030297
37,opp_shooting_Corner_3P%,0.029107
21,team_shooting_FG% by Distance_3-10,0.028228
7,team_per_game_FGA,0.028040


In [ ]:
# drop non-feature columns
X = filtered_df_v12.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.8
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_ORB ⬌ team_per_game_2PA | correlation = 0.832
team_per_game_BLK ⬌ team_per_game_FT% | correlation = 0.873
team_per_poss_FG ⬌ team_ORtg | correlation = 0.810
team_shooting_Corner_%3PA ⬌ team_per_poss_DRB | correlation = 0.848
opp_per_game_2P% ⬌ team_per_poss_DRB | correlation = 0.809
opp_shooting_% of FG Ast'd_3P ⬌ team_per_poss_DRB | correlation = 0.812


In [ ]:
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1]
}

In [ ]:
# subset just the league-wide data
df_team = filtered_df_v12.copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# define param grid
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1],
    "colsample_bytree": [0.8, 1]
}

# initialize base model
base_model = XGBRegressor(
    min_child_weight=1,
    random_state=42,
    verbosity=0
)

# run grid search with 3-fold CV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=1,
    n_jobs=-1
)

# fit search
grid_search.fit(X, y)

# best params
print("\nBest parameters found:")
print(grid_search.best_params_)

# best score
print(f"\nBest MAE (CV): {-grid_search.best_score_:.4f}")

Fitting 3 folds for each of 108 candidates, totalling 324 fits

Best parameters found:
{'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}

Best MAE (CV): 8.6212


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v12["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v12.copy()
    else:
        df_team = filtered_df_v12[filtered_df_v12["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.697418,29.302582,9.042065,5.302582,-0.848202
1,CHI,1.612991,21.142403,10.167221,10.361557,-0.394159
2,CON,1.908020,20.024239,9.117159,6.966129,-0.370147
3,DAL,1.112938,17.930550,7.881152,6.644501,-0.619384
4,IND,0.619118,14.455627,7.547647,8.694450,-0.153583
5,LAS,1.847069,20.152931,9.064523,8.000000,-0.099592
6,LVA,1.232620,18.232620,7.799897,7.000000,-0.380216
7,MIN,0.476959,27.371780,11.558397,10.890656,0.088494
8,NYL,2.971184,17.579269,6.759277,5.275227,0.291995
9,PHO,0.217506,19.994232,11.094264,9.717506,0.048197


In [ ]:
# models to train: 12 teams + league-wide
teams = filtered_df_v12["team"].unique().tolist() + ["League"]

# store results
results = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = filtered_df_v12.copy()
    else:
        df_team = filtered_df_v12[filtered_df_v12["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # model
    model = XGBRegressor(
        n_estimators=100,
        max_depth=3,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )

    # MAE (negated)
    neg_mae_scores = cross_val_score(model, X, y, scoring="neg_mean_absolute_error", cv=10)
    mae_scores = -neg_mae_scores

    # R²
    r2_scores = cross_val_score(model, X, y, scoring="r2", cv=10)

    # store both sets
    results.append({
        "Model": team_name,
        "MAE_min": mae_scores.min(),
        "MAE_max": mae_scores.max(),
        "MAE_mean": mae_scores.mean(),
        "MAE_median": np.median(mae_scores),
        "R2_min": r2_scores.min(),
        "R2_max": r2_scores.max(),
        "R2_mean": r2_scores.mean(),
        "R2_median": np.median(r2_scores)
    })

# create summary DataFrame
results_df = pd.DataFrame(results)
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2_min,R2_max,R2_mean,R2_median
0,ATL,3.508930,19.250000,8.104744,6.566471,-3.228904,-0.000144,-0.817832,-0.337204
1,CHI,2.061405,13.750000,7.922522,7.160944,-9.633927,-0.000001,-1.352348,-0.196554
2,CON,4.178896,13.000000,8.362984,8.262123,-4.199431,0.136269,-0.684993,-0.399390
3,DAL,1.250000,14.546692,9.082028,9.429706,-2.911178,0.014238,-0.755091,-0.214249
4,IND,3.073570,13.000000,8.157331,8.432936,-2.711883,0.201647,-0.962626,-0.628400
5,LAS,4.640919,17.774118,8.609738,8.054203,-1.410318,-0.001252,-0.462690,-0.278375
6,LVA,2.301861,12.706177,7.498015,7.078327,-3.020658,-0.001343,-0.988823,-0.851452
7,MIN,5.006443,11.943167,8.537713,8.934809,-2.460590,0.100193,-0.634334,-0.321766
8,NYL,5.648062,11.342678,7.581281,7.009244,-24.847496,0.396241,-2.576591,0.062027
9,PHO,5.601460,12.083982,8.875272,9.510556,-4.268547,0.268144,-0.485871,0.004899


In [36]:
# load the player game logs CSV from the data folder
df = pd.read_csv("LHL-final-final-project/data/all_persona_and_team_data.csv")


# preview
df.head()

,team,year,home_away,opp,team_score,opp_score,day,month,all_around_star,and_one_machine,...,team_3par,team_ts%,team_efg%,team_tov%,team_orb%,team_ft/fga,team_efg%.1,team_tov%.1,team_drb%,team_ft/fga.1
0,ATL,2020,1.0,DAL,105,95,26.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
1,ATL,2020,0.0,LVA,70,100,29.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
2,ATL,2020,1.0,NYL,84,78,31.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
3,ATL,2020,0.0,IND,77,93,2.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
4,ATL,2020,1.0,PHO,74,81,4.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231


In [37]:
# Apply transformations to match the column names structure
df.columns = df.columns.str.replace('%', '_pct') \
                       .str.replace('/', '_') \
                       .str.replace('.', '') \
                       .str.replace('-', '_') \
                       .str.replace('__', '_') \
                       .str.replace(' ', '_')

In [38]:
for col in df.columns:
    print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
team_per_game_fg
team_per_game_fga
team_per_game_fg_pct
team_per_game_3p
team_per_game_3pa
team_per_game_3p_pct
team_per_game_2p
team_per_game_2pa
team_per_game_2p_pct
team_per_game_ft
team_per_game_fta
team_per_game_ft_pct
team_per_game_orb
team_per_game_drb
team_per_game_trb
team_per_game_ast
team_per_game_stl
team_per_game_blk
team_per_game_tov
team_per_game_pf
team_per_game_pts
opp_per_game_fg
opp_per_game_fga
opp_per_game_fg_pct
opp_per_game_3p
opp_per_game_3pa
opp_per_game_3p_pct
opp_per_game_2p
opp_per_game_2pa
o

In [39]:
# models to train: 12 teams + league-wide
teams = df["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df.copy()
    else:
        df_team = df[df["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=1,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.061470,31.744766,8.219763,4.830376,0.024379
1,CHI,0.047607,24.162727,9.079511,7.837273,-0.026913
2,CON,0.081848,40.297462,7.478032,5.448677,-0.349860
3,DAL,0.695320,25.575737,9.553164,8.471611,-0.156328
4,IND,0.582161,19.256653,7.867328,7.743347,-0.064263
5,LAS,0.174484,20.843269,6.858389,5.917770,0.001546
6,MIN,0.265747,23.260307,8.297952,6.625893,-0.009744
7,NYL,0.216965,44.711250,8.912576,6.809303,0.031127
8,PHO,0.336731,24.543495,9.414747,7.567757,0.175986
9,SEA,0.678574,27.010536,10.568737,9.123840,-0.171631


In [40]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
49,team_per_game_pts,0.158814
91,team_per_poss_pts,0.089895
70,opp_per_game_pts,0.035548
44,team_per_game_ast,0.034096
122,team_fg_pct_by_distance_0_3,0.028738
...,...,...
162,team_orb_pct,0.000000
164,team_efg_pct1,0.000000
165,team_tov_pct1,0.000000
166,team_drb_pct,0.000000


In [41]:
# drop non-feature columns
X = df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.9999
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

offensive_hub ⬌ floor_general | correlation = 1.000
three_point_specialist ⬌ stretch_big | correlation = 1.000
team_per_poss_fg_pct ⬌ team_per_game_fg_pct | correlation = 1.000
team_per_poss_3p_pct ⬌ team_per_game_3p_pct | correlation = 1.000
team_per_poss_2p_pct ⬌ team_per_game_2p_pct | correlation = 1.000
team_per_poss_ft_pct ⬌ team_per_game_ft_pct | correlation = 1.000
opp_per_poss_fg_pct ⬌ opp_per_game_fg_pct | correlation = 1.000
opp_per_poss_3p_pct ⬌ opp_per_game_3p_pct | correlation = 1.000
opp_per_poss_2p_pct ⬌ opp_per_game_2p_pct | correlation = 1.000
opp_per_poss_ft_pct ⬌ opp_per_game_ft_pct | correlation = 1.000
team_fg_pct ⬌ team_per_game_fg_pct | correlation = 1.000
team_fg_pct ⬌ team_per_poss_fg_pct | correlation = 1.000
team_pct_of_fga_by_distance_3p ⬌ team_pct_of_fga_by_distance_2p | correlation = 1.000
team_fg_pct_by_distance_2p ⬌ team_per_game_2p_pct | correlation = 1.000
team_fg_pct_by_distance_2p ⬌ team_per_poss_2p_pct | correlation = 1.000
team_fg_pct_by_distance_3

In [42]:
# List of columns to drop
columns_to_drop = [
    "floor_general", "stretch_big", "team_per_game_fg_pct", "team_per_game_3p_pct",
    "team_per_game_2p_pct", "team_per_game_ft_pct", "opp_per_game_fg_pct", "opp_per_game_3p_pct",
    "opp_per_game_2p_pct", "opp_per_game_ft_pct", "team_fg_pct", "team_pct_of_fga_by_distance_2p",
    "team_fg_pct_by_distance_2p", "team_fg_pct_by_distance_3p", "opp_fg_pct",
    "opp_pct_of_fga_by_distance_2p", "opp_fg_pct_by_distance_2p", "opp_fg_pct_by_distance_3p",
    "team_ortg", "team_drtg", "team_pct_of_fga_by_distance_2p", "team_pct_of_fga_by_distance_3p"
]

# Create the filtered DataFrame
df_all_filtered_v1 = df.drop(columns=columns_to_drop)

# Verify changes
df_all_filtered_v1.head()

,team,year,home_away,opp,team_score,opp_score,day,month,all_around_star,and_one_machine,...,team_3par,team_ts_pct,team_efg_pct,team_tov_pct,team_orb_pct,team_ft_fga,team_efg_pct1,team_tov_pct1,team_drb_pct,team_ft_fga1
0,ATL,2020,1.0,DAL,105,95,26.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
1,ATL,2020,0.0,LVA,70,100,29.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
2,ATL,2020,1.0,NYL,84,78,31.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
3,ATL,2020,0.0,IND,77,93,2.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
4,ATL,2020,1.0,PHO,74,81,4.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231


In [43]:
# models to train: 12 teams + league-wide
teams = df_all_filtered_v1["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_all_filtered_v1.copy()
    else:
        df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # model
    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=1,
        colsample_bytree=1,
        min_child_weight=1,
        random_state=42,
        verbosity=0
    )
    model.fit(X_train, y_train)

    # predictions
    y_pred = model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred)
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2
0,ATL,0.061470,31.744766,8.219763,4.830376,0.024379
1,CHI,0.047607,24.162727,9.079511,7.837273,-0.026913
2,CON,0.081848,40.297462,7.478032,5.448677,-0.349860
3,DAL,0.695320,25.575737,9.553164,8.471611,-0.156328
4,IND,0.582161,19.256653,7.867328,7.743347,-0.064263
5,LAS,0.174484,20.843269,6.858389,5.917770,0.001546
6,MIN,0.265747,23.260307,8.297952,6.625893,-0.009744
7,NYL,0.216965,44.711250,8.912576,6.809303,0.031127
8,PHO,0.336731,24.543495,9.414747,7.567757,0.175986
9,SEA,0.678574,27.010536,10.568737,9.123840,-0.171631


In [44]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
43,team_per_game_pts,0.159179
81,team_per_poss_pts,0.090101
60,opp_per_game_pts,0.035629
38,team_per_game_ast,0.031608
108,team_fg_pct_by_distance_0_3,0.028804
...,...,...
141,team_orb_pct,0.000000
143,team_efg_pct1,0.000000
144,team_tov_pct1,0.000000
145,team_drb_pct,0.000000


In [45]:
# models to train: 12 teams + league-wide
teams = df_all_filtered_v1["team"].unique().tolist() + ["League"]

# store overall results summary
results = []

# store row-level predictions
all_predictions = []

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100, 300, 500],  # Number of boosting rounds
    'max_depth': [3, 5, 7],            # Maximum depth of the tree
    'learning_rate': [0.01, 0.05, 0.1],  # Step size
    'subsample': [0.8, 1],             # Fraction of samples to use
    'colsample_bytree': [0.8, 1],      # Fraction of features to use
    'min_child_weight': [1, 5]         # Minimum sum of instance weight (hessian) needed in a child
}

for team_name in teams:
    # subset data
    if team_name == "League":
        df_team = df_all_filtered_v1.copy()
    else:
        df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

    # features and target
    X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
    y = df_team["team_score"]

    # train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # meta info for row tracking
    meta = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)

    # XGBRegressor
    model = XGBRegressor(random_state=42, verbosity=0)

    # GridSearchCV for hyperparameter tuning
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1, scoring='neg_mean_absolute_error', verbose=1)
    grid_search.fit(X_train, y_train)

    # Get best parameters from GridSearchCV
    best_model = grid_search.best_estimator_

    # Print the best parameters
    print(f"Best params for {team_name}: {grid_search.best_params_}")

    # predictions
    y_pred = best_model.predict(X_test)

    # row-level error
    team_score_mae = np.abs(y_pred - meta["team_score"].values)

    # build prediction rows
    for i in range(len(meta)):
        all_predictions.append({
            "Model": team_name,
            "team": meta.loc[i, "team"],
            "opp": meta.loc[i, "opp"],
            "team_score": meta.loc[i, "team_score"],
            "team_score_pred": y_pred[i],
            "team_score_mae": team_score_mae[i],
        })

    # summary metrics
    results.append({
        "Model": team_name,
        "MAE_min": team_score_mae.min(),
        "MAE_max": team_score_mae.max(),
        "MAE_mean": team_score_mae.mean(),
        "MAE_median": np.median(team_score_mae),
        "R2": r2_score(y_test, y_pred),
        "Best Params": grid_search.best_params_
    })

# create DataFrames
predictions_df = pd.DataFrame(all_predictions)
results_df = pd.DataFrame(results)

# return full results
results_df

Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best params for ATL: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.8}
Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best params for CHI: {'colsample_bytree': 1, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 1}
Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best params for CON: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.8}
Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best params for DAL: {'colsample_bytree': 1, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 0.8}
Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best params for IND: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_we

,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,ATL,0.037270,30.506805,7.701218,4.744476,0.044650,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."
1,CHI,0.399269,24.399269,9.014057,8.121841,-0.012528,"{'colsample_bytree': 1, 'learning_rate': 0.01,..."
2,CON,0.100685,21.603088,7.506404,6.396912,-0.029421,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."
3,DAL,0.210495,27.641960,8.522603,7.601273,0.024613,"{'colsample_bytree': 1, 'learning_rate': 0.01,..."
4,IND,0.854927,21.974068,7.583070,6.858681,0.057143,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."
5,LAS,0.341209,20.783859,6.467800,5.783859,0.107972,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."
6,MIN,0.144836,22.981544,8.146520,6.586769,0.024754,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."
7,NYL,0.044266,31.400032,9.196253,8.347801,0.138567,"{'colsample_bytree': 1, 'learning_rate': 0.01,..."
8,PHO,0.601700,30.398300,10.166848,9.078270,0.122489,"{'colsample_bytree': 1, 'learning_rate': 0.01,..."
9,SEA,0.051132,23.162033,9.248378,8.789062,0.011706,"{'colsample_bytree': 1, 'learning_rate': 0.01,..."


In [48]:
# Pick team
team_name = "ATL"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.02],
    'subsample': [0.7],
    'colsample_bytree': [0.8],
    'min_child_weight': [5]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 27 candidates, totalling 81 fits
Best params for ATL: {'colsample_bytree': 0.8, 'learning_rate': 0.02, 'max_depth': 6, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.7}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,ATL,0.046783,29.532242,7.79878,5.532242,-0.01424,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [49]:
ATL_xgb_params = {
    'n_estimators': 100,
    'max_depth': 6,
    'learning_rate': 0.02,
    'subsample': 0.7,
    'colsample_bytree': 0.8,
    'min_child_weight': 5
}

In [51]:
# Pick team
team_name = "CHI"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [4],
    'learning_rate': [0.005],
    'subsample': [1],
    'colsample_bytree': [1],
    'min_child_weight': [1]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best params for CHI: {'colsample_bytree': 1, 'learning_rate': 0.005, 'max_depth': 4, 'min_child_weight': 1, 'n_estimators': 100, 'subsample': 1}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,CHI,0.415894,27.116592,8.843388,7.266243,-0.393422,"{'colsample_bytree': 1, 'learning_rate': 0.005..."


In [54]:
CHI_xgb_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.005,
    'subsample': 1,
    'colsample_bytree': 1,
    'min_child_weight': 1
}

In [55]:
# Pick team
team_name = "CON"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [200],
    'max_depth': [4],
    'learning_rate': [0.02],
    'subsample': [0.7],
    'colsample_bytree': [0.9],
    'min_child_weight': [4]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best params for CON: {'colsample_bytree': 0.9, 'learning_rate': 0.02, 'max_depth': 4, 'min_child_weight': 4, 'n_estimators': 200, 'subsample': 0.7}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,CON,0.045204,26.059441,8.325805,8.0,-0.045382,"{'colsample_bytree': 0.9, 'learning_rate': 0.0..."


In [56]:
CON_xgb_params = {
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.02,
    'subsample': 0.7,
    'colsample_bytree': 0.9,
    'min_child_weight': 4
}

In [59]:
# Pick team
team_name = "DAL"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [2],
    'learning_rate': [0.002],
    'subsample': [0.7],
    'colsample_bytree': [0.9],
    'min_child_weight': [3]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best params for DAL: {'colsample_bytree': 0.9, 'learning_rate': 0.002, 'max_depth': 2, 'min_child_weight': 3, 'n_estimators': 100, 'subsample': 0.7}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,DAL,0.042244,28.98716,8.971536,8.532917,-0.001275,"{'colsample_bytree': 0.9, 'learning_rate': 0.0..."


In [60]:
DAL_xgb_params = {
    'n_estimators': 100,
    'max_depth': 2,
    'learning_rate': 0.002,
    'subsample': 0.7,
    'colsample_bytree': 0.9,
    'min_child_weight': 3
}

In [61]:
# Pick team
team_name = "IND"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [2],
    'learning_rate': [0.01],
    'subsample': [0.9],
    'colsample_bytree': [0.8],
    'min_child_weight': [4]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 432 candidates, totalling 1296 fits
Best params for IND: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 2, 'min_child_weight': 4, 'n_estimators': 100, 'subsample': 0.9}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,IND,0.049843,29.751617,8.924211,8.034981,-0.241047,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [62]:
IND_xgb_params = {
    'n_estimators': 100,
    'max_depth': 2,
    'learning_rate': 0.01,
    'subsample': 0.9,
    'colsample_bytree': 0.8,
    'min_child_weight': 4
}

In [66]:
LAS_xgb_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.005,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'min_child_weight': 5
}

In [69]:
# Pick team
team_name = "MIN"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [2],
    'learning_rate': [0.003],
    'subsample': [0.8],
    'colsample_bytree': [0.9],
    'min_child_weight': [10]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best params for MIN: {'colsample_bytree': 0.9, 'learning_rate': 0.003, 'max_depth': 2, 'min_child_weight': 10, 'n_estimators': 100, 'subsample': 0.8}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,MIN,0.339905,30.771454,8.502394,6.926788,-0.001096,"{'colsample_bytree': 0.9, 'learning_rate': 0.0..."


In [71]:
MIN_xgb_params = {
    'n_estimators': 100,
    'max_depth': 2,
    'learning_rate': 0.003,
    'subsample': 0.8,
    'colsample_bytree': 0.9,
    'min_child_weight': 10
}

In [72]:
# Pick team
team_name = "NYL"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.005],
    'subsample': [1],
    'colsample_bytree': [1],
    'min_child_weight': [2]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 9 candidates, totalling 27 fits
Best params for NYL: {'colsample_bytree': 1, 'learning_rate': 0.005, 'max_depth': 6, 'min_child_weight': 2, 'n_estimators': 100, 'subsample': 1}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,NYL,0.204102,26.795898,9.810541,9.506805,-0.405975,"{'colsample_bytree': 1, 'learning_rate': 0.005..."


In [74]:
NYL_xgb_params = {
    'n_estimators': 100,
    'max_depth': 6,
    'learning_rate': 0.005,
    'subsample': 1,
    'colsample_bytree': 1,
    'min_child_weight': 2
}

In [75]:
# Pick team
team_name = "PHO"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [200],
    'max_depth': [4],
    'learning_rate': [0.01],
    'subsample': [1],
    'colsample_bytree': [1],
    'min_child_weight': [4]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best params for PHO: {'colsample_bytree': 1, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 4, 'n_estimators': 200, 'subsample': 1}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,PHO,0.561729,31.657303,11.45441,9.386234,-0.239897,"{'colsample_bytree': 1, 'learning_rate': 0.01,..."


In [77]:
PHO_xgb_params = {
    'n_estimators': 200,
    'max_depth': 4,
    'learning_rate': 0.01,
    'subsample': 1,
    'colsample_bytree': 1,
    'min_child_weight': 4
}

In [80]:
# Pick team
team_name = "SEA"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [5],
    'learning_rate': [0.001],
    'subsample': [1],
    'colsample_bytree': [1],
    'min_child_weight': [4]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best params for SEA: {'colsample_bytree': 1, 'learning_rate': 0.001, 'max_depth': 5, 'min_child_weight': 4, 'n_estimators': 100, 'subsample': 1}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,SEA,0.031395,20.011421,7.585017,6.880756,-0.015932,"{'colsample_bytree': 1, 'learning_rate': 0.001..."


In [82]:
SEA_xgb_params = {
    'n_estimators': 100,
    'max_depth': 5,
    'learning_rate': 0.001,
    'subsample': 1,
    'colsample_bytree': 1,
    'min_child_weight': 4
}

In [83]:
# Pick team
team_name = "WAS"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [4],
    'learning_rate': [0.01],
    'subsample': [1],
    'colsample_bytree': [1],
    'min_child_weight': [6]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best params for WAS: {'colsample_bytree': 1, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 6, 'n_estimators': 100, 'subsample': 1}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,WAS,0.030945,19.900787,6.954247,7.019859,0.052284,"{'colsample_bytree': 1, 'learning_rate': 0.01,..."


In [85]:
WAS_xgb_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.01,
    'subsample': 1,
    'colsample_bytree': 1,
    'min_child_weight': 6
}

In [87]:
# Pick team
team_name = "LVA"

# subset data for the selected team
df_team = df_all_filtered_v1[df_all_filtered_v1["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100],
    'max_depth': [4],
    'learning_rate': [0.004],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'min_child_weight': [5]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best params for LVA: {'colsample_bytree': 0.8, 'learning_rate': 0.004, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.8}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,LVA,0.187836,25.50853,7.6897,5.848183,-0.187597,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [89]:
LVA_xgb_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.004,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5
}

In [91]:
# Pick team
team_name = "League"

# subset data for the selected team
df_team = df_all_filtered_v1.copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [300],
    'max_depth': [2],
    'learning_rate': [0.02, 0.03, 0.04],
    'subsample': [1],
    'colsample_bytree': [1],
    'min_child_weight': [1]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best params for League: {'colsample_bytree': 1, 'learning_rate': 0.02, 'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 1}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,League,0.021866,35.505775,8.2623,7.009895,0.061792,"{'colsample_bytree': 1, 'learning_rate': 0.02,..."


In [92]:
League_xgb_params = {
    'n_estimators': 300,
    'max_depth': 2,
    'learning_rate': 0.02,
    'subsample': 1,
    'colsample_bytree': 1,
    'min_child_weight': 1
}

In [93]:
# Combine params for all teams + League
all_team_xgb_team_score_params = {
    'ATL': ATL_xgb_params,
    'CHI': CHI_xgb_params,
    'CON': CON_xgb_params,
    'DAL': DAL_xgb_params,
    'IND': IND_xgb_params,
    'LAS': LAS_xgb_params,
    'MIN': MIN_xgb_params,
    'NYL': NYL_xgb_params,
    'PHO': PHO_xgb_params,
    'SEA': SEA_xgb_params,
    'WAS': WAS_xgb_params,
    'LVA': LVA_xgb_params,
    'League': League_xgb_params
}

# Save as JSON locally in Colab
import json

with open("all_team_xgb_team_score_params.json", "w") as f:
    json.dump(all_team_xgb_team_score_params, f, indent=4)

# Download to your local machine
from google.colab import files
files.download("all_team_xgb_team_score_params.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [94]:
# save to CSV
df_all_filtered_v1.to_csv("team_score_xgb_filtered_v1.csv", index=False)

# download to local machine
from google.colab import files
files.download("team_score_xgb_filtered_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [96]:
# Define your team
team_name = "ATL"
model_df = df_all_filtered_v1

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,ATL,7.483708,6.567657,0.186665,7.759212,4.919464,0.049791


In [97]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
58,opp_per_game_tov,0.103352
47,opp_per_game_3pa,0.079356
33,team_per_game_ft,0.073292
35,team_per_game_orb,0.068128
34,team_per_game_fta,0.061385
...,...,...
142,team_ft_fga,0.000000
143,team_efg_pct1,0.000000
144,team_tov_pct1,0.000000
145,team_drb_pct,0.000000


In [99]:
# get features to drop from df_importances
features_to_drop = df_importances[df_importances["Importance_team_score"] <= 0.001]["Feature"].tolist()

# create new filtered dataframe
df_all_filtered_v2 = df_all_filtered_v1.drop(columns=features_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_all_filtered_v2.shape}")

Dropped 116 features. New shape: (2040, 37)


In [100]:
# Define your team
team_name = "ATL"
model_df = df_all_filtered_v2

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,ATL,7.481202,6.415146,0.188488,7.83282,4.923141,0.040353


In [101]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
28,team_per_game_tov,0.129189
24,team_per_game_ft,0.092787
27,team_per_game_blk,0.063008
30,opp_per_game_tov,0.051783
3,catch_and_shoot,0.049277
26,team_per_game_orb,0.047951
22,team_per_game_3pa,0.046279
18,volume_shooter,0.046014
19,team_per_game_fg,0.041695
21,team_per_game_3p,0.041559


In [103]:
# Pick team
team_name = "ATL"
model_df = df_all_filtered_v2

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [4, 5, 6]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 486 candidates, totalling 1458 fits
Best params for ATL: {'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 200, 'subsample': 0.6}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,ATL,0.076965,28.760147,7.757323,5.713249,-0.025098,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [105]:
# drop non-feature columns
X = df_all_filtered_v2.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.9
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_3pa ⬌ team_per_game_3p | correlation = 0.940
team_per_game_fta ⬌ team_per_game_ft | correlation = 0.942


In [106]:
df_all_filtered_v3 = df_all_filtered_v2.drop(columns=[
    "team_per_game_3p",
    "team_per_game_ft"
])

In [107]:
# Define your team
team_name = "ATL"
model_df = df_all_filtered_v3

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,ATL,7.473219,6.328743,0.18908,7.813302,5.103043,0.042845


In [108]:
# Identify features to drop (importance < 0.01)
features_to_drop_v4 = df_importances[df_importances["Importance_team_score"] < 0.01]["Feature"].tolist()

# Create next version of filtered dataframe
df_atl_filtered_v4 = df_all_filtered_v3.drop(columns=features_to_drop_v4)

# Quick verification
print(f"Dropped {len(features_to_drop_v4)} features. New shape: {df_atl_filtered_v4.shape}")

Dropped 6 features. New shape: (2040, 29)


In [109]:
# Define your team
team_name = "ATL"
model_df = df_atl_filtered_v4

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,ATL,7.482063,6.382111,0.18769,7.831898,5.0867,0.040961


In [110]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,team_per_game_fta,0.108944
19,team_per_game_orb,0.080756
20,team_per_game_blk,0.078645
22,opp_per_game_tov,0.078006
3,catch_and_shoot,0.060731
14,volume_shooter,0.055168
17,team_per_game_3pa,0.047552
7,heave_chucker,0.047042
11,self_creator,0.044552
15,team_per_game_fg,0.041540


In [111]:
# Pick team
team_name = "ATL"
model_df = df_atl_filtered_v4

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# Split data based on year (test = 2024, train = everything else)
train_df = df_team[df_team["year"] != 2024]
test_df = df_team[df_team["year"] == 2024]

# features and target
X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]
X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# Define hyperparameters for GridSearchCV
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 6, 7],
    'learning_rate': [0.01, 0.02, 0.03],
    'subsample': [0.6, 0.7, 0.8],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [4, 5, 6]
}

# GridSearchCV for hyperparameter tuning
grid_search = GridSearchCV(
    estimator=XGBRegressor(random_state=42, verbosity=0),
    param_grid=param_grid,
    cv=3,
    n_jobs=-1,
    scoring='neg_mean_absolute_error',
    verbose=1
)
grid_search.fit(X_train, y_train)

# Best model and params
best_model = grid_search.best_estimator_
print(f"Best params for {team_name}: {grid_search.best_params_}")

# Predictions and evaluation
y_pred = best_model.predict(X_test)

team_score_mae = np.abs(y_pred - y_test.values)

# Summary results
results = {
    "Model": team_name,
    "MAE_min": team_score_mae.min(),
    "MAE_max": team_score_mae.max(),
    "MAE_mean": team_score_mae.mean(),
    "MAE_median": np.median(team_score_mae),
    "R2": r2_score(y_test, y_pred),
    "Best Params": grid_search.best_params_
}

# Display results
results_df = pd.DataFrame([results])
results_df

Fitting 3 folds for each of 486 candidates, totalling 1458 fits
Best params for ATL: {'colsample_bytree': 0.8, 'learning_rate': 0.02, 'max_depth': 5, 'min_child_weight': 4, 'n_estimators': 100, 'subsample': 0.6}


,Model,MAE_min,MAE_max,MAE_mean,MAE_median,R2,Best Params
0,ATL,0.091133,28.778976,7.731701,6.0,-0.015334,"{'colsample_bytree': 0.8, 'learning_rate': 0.0..."


In [114]:
from sklearn.ensemble import RandomForestRegressor

# Define your team
team_name = "ATL"
model_df = df_atl_filtered_v4

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# define and train Random Forest model (simple initial params)
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# make predictions
y_pred_test_rf = rf_model.predict(X_test)
y_pred_train_rf = rf_model.predict(X_train)

# calculate row-level MAE
team_score_mae_test_rf = np.abs(y_pred_test_rf - y_test.values)
team_score_mae_train_rf = np.abs(y_pred_train_rf - y_train.values)

# store prediction rows (test)
predictions_test_rf_df = pd.DataFrame({
    "Model": f"{team_name}_RF",
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test_rf,
    "team_score_mae": team_score_mae_test_rf
})

# store prediction rows (train)
predictions_train_rf_df = pd.DataFrame({
    "Model": f"{team_name}_RF",
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train_rf,
    "team_score_mae": team_score_mae_train_rf
})

# summary metrics (train and test)
results_rf_df = pd.DataFrame([{
    "Model": f"{team_name}_RF",
    "Train_MAE_mean": team_score_mae_train_rf.mean(),
    "Train_MAE_median": np.median(team_score_mae_train_rf),
    "Train_R2": r2_score(y_train, y_pred_train_rf),
    "Test_MAE_mean": team_score_mae_test_rf.mean(),
    "Test_MAE_median": np.median(team_score_mae_test_rf),
    "Test_R2": r2_score(y_test, y_pred_test_rf)
}])

# display summary metrics
results_rf_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,ATL_RF,7.582407,6.387296,0.167171,7.772846,4.731172,0.048487


In [116]:
# define RF param grid
param_grid_rf = {
    'n_estimators': [100],
    'max_depth': [6],
    'min_samples_leaf': [6, 7, 8],
    'max_features': ['sqrt']
}

# grid search setup
grid_search_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

# run grid search
grid_search_rf.fit(X_train, y_train)

# output best parameters clearly
print(f"Best RF params for {team_name}: {grid_search_rf.best_params_}")

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best RF params for ATL: {'max_depth': 6, 'max_features': 'sqrt', 'min_samples_leaf': 7, 'n_estimators': 100}


In [119]:
from sklearn.ensemble import RandomForestRegressor

# Define your team
team_name = "ATL"
model_df = df_atl_filtered_v4

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# define and train Random Forest model (simple initial params)
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=6,
    max_features='sqrt',
    min_samples_leaf=7,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# make predictions
y_pred_test_rf = rf_model.predict(X_test)
y_pred_train_rf = rf_model.predict(X_train)

# calculate row-level MAE
team_score_mae_test_rf = np.abs(y_pred_test_rf - y_test.values)
team_score_mae_train_rf = np.abs(y_pred_train_rf - y_train.values)

# store prediction rows (test)
predictions_test_rf_df = pd.DataFrame({
    "Model": f"{team_name}_RF",
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test_rf,
    "team_score_mae": team_score_mae_test_rf
})

# store prediction rows (train)
predictions_train_rf_df = pd.DataFrame({
    "Model": f"{team_name}_RF",
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train_rf,
    "team_score_mae": team_score_mae_train_rf
})

# summary metrics (train and test)
results_rf_df = pd.DataFrame([{
    "Model": f"{team_name}_RF",
    "Train_MAE_mean": team_score_mae_train_rf.mean(),
    "Train_MAE_median": np.median(team_score_mae_train_rf),
    "Train_R2": r2_score(y_train, y_pred_train_rf),
    "Test_MAE_mean": team_score_mae_test_rf.mean(),
    "Test_MAE_median": np.median(team_score_mae_test_rf),
    "Test_R2": r2_score(y_test, y_pred_test_rf)
}])

# display summary metrics
results_rf_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,ATL_RF,7.784119,6.53298,0.105095,7.766246,5.074177,0.036081


In [122]:
# extract best RF model
best_rf_model = grid_search_rf.best_estimator_

# get feature importances
rf_importances = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": best_rf_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

# inspect top features
rf_importances.head(40)

,Feature,Importance
7,heave_chucker,0.171054
1,home_away,0.114234
19,team_per_game_orb,0.096683
18,team_per_game_fta,0.078048
17,team_per_game_3pa,0.067010
3,catch_and_shoot,0.053024
15,team_per_game_fg,0.052318
21,team_per_game_tov,0.045706
14,volume_shooter,0.042063
22,opp_per_game_tov,0.041313


In [126]:
# ATL average baseline
atl_avg_score = y_train.mean()
baseline_preds = np.full_like(y_test, atl_avg_score)

# Baseline metrics
baseline_mae = np.abs(baseline_preds - y_test.values)

# XGB metrics (assuming y_pred_test from your XGB model)
xgb_mae = np.abs(y_pred_test - y_test.values)

print("🎯 **XGB Model Metrics** 🎯")
print(f"MAE Min: {xgb_mae.min():.3f}")
print(f"MAE Max: {xgb_mae.max():.3f}")
print(f"MAE Mean: {xgb_mae.mean():.3f}")
print(f"MAE Median: {np.median(xgb_mae):.3f}")
print(f"R²: {r2_score(y_test, y_pred_test):.3f}")

print("\n📌 **ATL Avg. Baseline Metrics** 📌")
print(f"MAE Min: {baseline_mae.min():.3f}")
print(f"MAE Max: {baseline_mae.max():.3f}")
print(f"MAE Mean: {baseline_mae.mean():.3f}")
print(f"MAE Median: {np.median(baseline_mae):.3f}")
print(f"R²: {r2_score(y_test, baseline_preds):.3f}")

🎯 **XGB Model Metrics** 🎯
MAE Min: 0.034
MAE Max: 30.944
MAE Mean: 7.832
MAE Median: 5.087
R²: 0.041

📌 **ATL Avg. Baseline Metrics** 📌
MAE Min: 0.000
MAE Max: 29.000
MAE Mean: 8.023
MAE Median: 6.000
R²: -0.013


In [124]:
# Save your filtered dataframe to CSV
df_atl_filtered_v4.to_csv("atl_team_score_xgb_df.csv", index=False)

# Download the CSV file to your local machine
from google.colab import files
files.download("atl_team_score_xgb_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
# load the player game logs CSV from the data folder
df_all_filtered_v1 = pd.read_csv("LHL-final-final-project/data/team_score_xgb_filtered_v1.csv")


# preview
df_all_filtered_v1.head()

,team,year,home_away,opp,team_score,opp_score,day,month,all_around_star,and_one_machine,...,team_3par,team_ts_pct,team_efg_pct,team_tov_pct,team_orb_pct,team_ft_fga,team_efg_pct1,team_tov_pct1,team_drb_pct,team_ft_fga1
0,ATL,2020,1.0,DAL,105,95,26.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
1,ATL,2020,0.0,LVA,70,100,29.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
2,ATL,2020,1.0,NYL,84,78,31.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
3,ATL,2020,0.0,IND,77,93,2.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
4,ATL,2020,1.0,PHO,74,81,4.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231


In [49]:
# Save your filtered dataframe to CSV
df_all_filtered_v1.to_csv("all_filtered_v1_xgb_df.csv", index=False)

# Download the CSV file to your local machine
from google.colab import files
files.download("all_filtered_v1_xgb_df.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
# clearly define columns to drop
cols_to_drop = [
    "defensive_anchor",
    "defensive_rebounder",
    "rim_protector",
    "team_per_game_drb",
    "team_per_game_trb",
    "team_per_game_stl",
    "team_per_game_blk",
    "opp_per_game_ast",
    "opp_per_game_trb",
    "opp_per_game_orb",
    "opp_per_game_drb",
    "opp_per_game_ft",
    "opp_per_game_2p",
    "team_per_poss_drb",
    "team_per_poss_trb",
    "team_per_poss_stl",
    "team_per_poss_blk",
    "opp_per_poss_orb",
    "opp_per_poss_drb",
    "opp_per_poss_trb",
    "opp_per_poss_ast",
    "opp_dist",
    "opp_pct_of_fga_by_distance_0_3",
    "opp_pct_of_fga_by_distance_3_10",
    "opp_pct_of_fga_by_distance_10_16",
    "opp_pct_of_fga_by_distance_16_3p",
    "opp_pct_of_fga_by_distance_3p",
    "opp_fg_pct_by_distance_0_3",
    "opp_fg_pct_by_distance_3_10",
    "opp_fg_pct_by_distance_10_16",
    "opp_fg_pct_by_distance_16_3p",
    "opp_pct_of_fg_ast'd_2p",
    "opp_pct_of_fg_ast'd_3p",
    "opp_corner_pct3pa",
    "opp_corner_3p_pct",
    "team_nrtg"
]

# drop columns clearly
df_team_score_filtered_v1 = df_all_filtered_v1.drop(columns=cols_to_drop)

In [51]:
df_team_score_filtered_v1.shape

(2040, 117)

In [66]:
# Save your filtered dataframe to CSV
df_team_score_filtered_v1.to_csv("team_score_xgb_filtered_v1.csv", index=False)

# Download the CSV file to your local machine
from google.colab import files
files.download("team_score_xgb_filtered_v1.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
# load JSON from provided path
with open('LHL-final-final-project/configs/all_team_xgb_team_score_params.json', 'r') as file:
    all_team_xgb_team_score_params = json.load(file)

# verify the data loaded correctly
print(all_team_xgb_team_score_params)

{'ATL': {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.02, 'subsample': 0.7, 'colsample_bytree': 0.8, 'min_child_weight': 5}, 'CHI': {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.005, 'subsample': 1, 'colsample_bytree': 1, 'min_child_weight': 1}, 'CON': {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.02, 'subsample': 0.7, 'colsample_bytree': 0.9, 'min_child_weight': 4}, 'DAL': {'n_estimators': 100, 'max_depth': 2, 'learning_rate': 0.002, 'subsample': 0.7, 'colsample_bytree': 0.9, 'min_child_weight': 3}, 'IND': {'n_estimators': 100, 'max_depth': 2, 'learning_rate': 0.01, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_weight': 4}, 'LAS': {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.005, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 5}, 'MIN': {'n_estimators': 100, 'max_depth': 2, 'learning_rate': 0.003, 'subsample': 0.8, 'colsample_bytree': 0.9, 'min_child_weight': 10}, 'NYL': {'n_estimators': 100, 'max_depth': 6, 'learnin

In [12]:
# CHI average baseline
chi_avg_score = y_train.mean()
baseline_preds = np.full_like(y_test, chi_avg_score)

# Baseline metrics
baseline_mae = np.abs(baseline_preds - y_test.values)

# # XGB metrics (assuming y_pred_test from your XGB model)
# xgb_mae = np.abs(y_pred_test - y_test.values)

# print("🎯 **XGB Model Metrics** 🎯")
# print(f"MAE Min: {xgb_mae.min():.3f}")
# print(f"MAE Max: {xgb_mae.max():.3f}")
# print(f"MAE Mean: {xgb_mae.mean():.3f}")
# print(f"MAE Median: {np.median(xgb_mae):.3f}")
# print(f"R²: {r2_score(y_test, y_pred_test):.3f}")

print("\n📌 **CHI Avg. Baseline Metrics** 📌")
print(f"MAE Min: {baseline_mae.min():.3f}")
print(f"MAE Max: {baseline_mae.max():.3f}")
print(f"MAE Mean: {baseline_mae.mean():.3f}")
print(f"MAE Median: {np.median(baseline_mae):.3f}")
print(f"R²: {r2_score(y_test, baseline_preds):.3f}")


📌 **CHI Avg. Baseline Metrics** 📌
MAE Min: 0.000
MAE Max: 22.000
MAE Mean: 9.023
MAE Median: 8.000
R²: -0.001


In [76]:
# Define your team
team_name = "CHI"
model_df = df_team_score_filtered_v1

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,CHI,8.35054,7.300713,0.122546,9.053587,7.699287,-0.005774


In [77]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
18,slasher,0.471805
23,team_per_game_fg,0.216977
0,home_away,0.096314
4,corner_3_specialist,0.091303
20,three_point_specialist,0.072869
...,...,...
105,team_ft_fga,0.000000
106,team_efg_pct1,0.000000
107,team_tov_pct1,0.000000
108,team_drb_pct,0.000000


In [78]:
# get features to drop from df_importances
features_to_drop = df_importances[df_importances["Importance_team_score"] <= 0.001]["Feature"].tolist()

# create new filtered dataframe
df_chi_filtered_v2 = df_team_score_filtered_v1.drop(columns=features_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_chi_filtered_v2.shape}")

Dropped 103 features. New shape: (2040, 14)


In [79]:
# Define your team
team_name = "CHI"
model_df = df_chi_filtered_v2

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,CHI,8.35054,7.300713,0.122546,9.053587,7.699287,-0.005774


In [80]:
# drop non-feature columns
X = df_chi_filtered_v2.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.2
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

plus_minus_driver ⬌ and_one_machine | correlation = 0.534
slasher ⬌ and_one_machine | correlation = 0.232
three_point_specialist ⬌ and_one_machine | correlation = 0.508
three_point_specialist ⬌ plus_minus_driver | correlation = 0.239


In [81]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
4,slasher,0.471805
6,team_per_game_fg,0.216977
0,home_away,0.096314
2,corner_3_specialist,0.091303
5,three_point_specialist,0.072869
1,and_one_machine,0.030318
3,plus_minus_driver,0.020413


In [82]:
# drop duplicates if accidentally listed more than once (cleaned up clearly above)
df_chi_filtered_v3 = df_chi_filtered_v2.drop(columns="corner_3_specialist")

In [83]:
# Define your team
team_name = "CHI"
model_df = df_chi_filtered_v3

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,CHI,8.374486,7.385025,0.119959,9.051885,7.699287,0.004835


In [84]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
3,slasher,0.553370
5,team_per_game_fg,0.189672
0,home_away,0.112599
4,three_point_specialist,0.082361
1,and_one_machine,0.038776
2,plus_minus_driver,0.023221


In [86]:
# drop duplicates if accidentally listed more than once (cleaned up clearly above)
df_chi_filtered_v4 = df_chi_filtered_v3.drop(columns="plus_minus_driver")

In [87]:
# Define your team
team_name = "CHI"
model_df = df_chi_filtered_v4

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,CHI,8.374486,7.385025,0.119959,9.051885,7.699287,0.004835


In [88]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
2,slasher,0.591973
4,team_per_game_fg,0.202903
0,home_away,0.120454
3,three_point_specialist,0.043188
1,and_one_machine,0.041481


In [89]:
# drop duplicates if accidentally listed more than once (cleaned up clearly above)
df_chi_filtered_v5 = df_chi_filtered_v4.drop(columns="and_one_machine")

In [90]:
# Define your team
team_name = "CHI"
model_df = df_chi_filtered_v5

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,CHI,8.380611,7.385025,0.116853,8.966322,7.699287,0.013867


In [91]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,slasher,0.579936
3,team_per_game_fg,0.230023
0,home_away,0.136554
2,three_point_specialist,0.053487


In [92]:
# Confirm clearly if features exist and add them back (if they were previously removed)
features_to_reintroduce = ["team_pace"]
df_chi_filtered_v6 = df_chi_filtered_v5.copy()

for feature in features_to_reintroduce:
    if feature not in df_chi_filtered_v6.columns and feature in df_all_filtered_v1.columns:
        df_chi_filtered_v6[feature] = df_all_filtered_v1.loc[df_chi_filtered_v6.index, feature]

In [93]:
# Define your team
team_name = "CHI"
model_df = df_chi_filtered_v6

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,CHI,8.356193,7.300713,0.119686,9.010479,7.699287,-0.010005


In [94]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,slasher,0.479540
3,team_per_game_fg,0.250232
0,home_away,0.113773
4,team_pace,0.112188
2,three_point_specialist,0.044268


In [95]:
# Confirm clearly if features exist and add them back (if they were previously removed)
features_to_reintroduce = ["team_efg_pct"]
df_chi_filtered_v7 = df_chi_filtered_v6.copy()

for feature in features_to_reintroduce:
    if feature not in df_chi_filtered_v7.columns and feature in df_all_filtered_v1.columns:
        df_chi_filtered_v7[feature] = df_all_filtered_v1.loc[df_chi_filtered_v7.index, feature]

In [96]:
# Define your team
team_name = "CHI"
model_df = df_chi_filtered_v7

# subset data for the selected team
df_team = model_df[model_df["team"] == team_name].copy()

# features and target
X = df_team.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y = df_team["team_score"]

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# meta info for tracking predictions
meta_test = df_team.loc[y_test.index, ["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = df_team.loc[y_train.index, ["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,CHI,8.356193,7.300713,0.119686,9.010479,7.699287,-0.010005


In [97]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,slasher,0.479540
3,team_per_game_fg,0.250232
0,home_away,0.113773
4,team_pace,0.112188
2,three_point_specialist,0.044268
5,team_efg_pct,0.000000


In [99]:
# League average baseline
league_avg_score = y_train.mean()
baseline_preds = np.full_like(y_test, league_avg_score)

# Baseline metrics
baseline_mae = np.abs(baseline_preds - y_test.values)

# # XGB metrics (assuming y_pred_test from your XGB model)
# xgb_mae = np.abs(y_pred_test - y_test.values)

# print("🎯 **XGB Model Metrics** 🎯")
# print(f"MAE Min: {xgb_mae.min():.3f}")
# print(f"MAE Max: {xgb_mae.max():.3f}")
# print(f"MAE Mean: {xgb_mae.mean():.3f}")
# print(f"MAE Median: {np.median(xgb_mae):.3f}")
# print(f"R²: {r2_score(y_test, y_pred_test):.3f}")

print("\n📌 **League Avg. Baseline Metrics** 📌")
print(f"MAE Min: {baseline_mae.min():.3f}")
print(f"MAE Max: {baseline_mae.max():.3f}")
print(f"MAE Mean: {baseline_mae.mean():.3f}")
print(f"MAE Median: {np.median(baseline_mae):.3f}")
print(f"R²: {r2_score(y_test, baseline_preds):.3f}")


📌 **League Avg. Baseline Metrics** 📌
MAE Min: 0.000
MAE Max: 35.000
MAE Mean: 8.537
MAE Median: 7.000
R²: -0.001


In [98]:
team_name = "League"
model_df = df_team_score_filtered_v1

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.989482,6.659973,0.163514,8.267972,7.008617,0.05811


In [100]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
35,team_per_game_pts,0.249111
32,team_per_game_ast,0.137511
33,team_per_game_tov,0.078793
55,team_per_poss_2p_pct,0.061237
91,team_pct_of_fg_ast'd_3p,0.047269
...,...,...
105,team_ft_fga,0.000000
106,team_efg_pct1,0.000000
107,team_tov_pct1,0.000000
108,team_drb_pct,0.000000


In [101]:
# get features to drop from df_importances
features_to_drop = df_importances[df_importances["Importance_team_score"] <= 0.001]["Feature"].tolist()

# create new filtered dataframe
df_league_filtered_v2 = df_team_score_filtered_v1.drop(columns=features_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v2.shape}")

Dropped 79 features. New shape: (2040, 38)


In [102]:
team_name = "League"
model_df = df_league_filtered_v2

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.989482,6.659973,0.163514,8.267972,7.008617,0.05811


In [104]:
# drop non-feature columns
X = df_league_filtered_v2.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])

# compute pairwise correlations
corr_matrix = X.corr().abs()

# select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# find feature pairs with high correlation
high_corr_pairs = [
    (col, row, upper.loc[row, col])
    for col in upper.columns
    for row in upper.index
    if upper.loc[row, col] > 0.9
]

# display highly correlated feature pairs
for f1, f2, corr in high_corr_pairs:
    print(f"{f1} ⬌ {f2} | correlation = {corr:.3f}")

team_per_game_pts ⬌ team_per_game_fg | correlation = 0.903
team_per_poss_orb ⬌ team_per_game_orb | correlation = 0.994
team_per_poss_pts ⬌ team_per_game_pts | correlation = 0.935
team_sos ⬌ team_mov | correlation = 0.959


In [105]:
# clearly defined columns to drop (duplicates removed)
cols_to_drop = [
    "team_per_game_orb",
    "team_per_game_pts"
]

# drop clearly and create new DataFrame
df_league_filtered_v3 = df_league_filtered_v2.drop(columns=cols_to_drop)

In [106]:
team_name = "League"
model_df = df_league_filtered_v3

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.997328,6.662899,0.161871,8.270497,7.126492,0.053196


In [109]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
13,team_per_game_fg,0.201193
22,team_per_poss_pts,0.155163
19,team_per_poss_fg_pct,0.086305
25,team_corner_3p_pct,0.071518
16,team_per_game_ast,0.069738
17,team_per_game_tov,0.049706
20,team_per_poss_2p_pct,0.049278
28,team_pace,0.035789
24,team_pct_of_fg_ast'd_3p,0.035612
14,team_per_game_fga,0.027801


In [110]:
# get features to drop from df_importances
features_to_drop = df_importances[df_importances["Importance_team_score"] <= 0.01]["Feature"].tolist()

# create new filtered dataframe
df_league_filtered_v4 = df_league_filtered_v3.drop(columns=features_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v4.shape}")

Dropped 9 features. New shape: (2040, 27)


In [111]:
team_name = "League"
model_df = df_league_filtered_v4

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.019664,6.553909,0.155477,8.3028,7.32901,0.052433


In [112]:
# clearly identify columns containing 'per_game'
cols_to_drop = [col for col in df_team_score_filtered_v1.columns if 'per_game' in col]

# clearly drop these columns to create a new DataFrame
df_league_filtered_v5 = df_team_score_filtered_v1.drop(columns=cols_to_drop)

In [113]:
team_name = "League"
model_df = df_league_filtered_v5

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.01071,6.690102,0.160804,8.27407,7.272575,0.054287


In [114]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
23,team_per_poss_fg,0.197045
72,team_sos,0.136270
39,team_per_poss_pts,0.112060
77,team_ts_pct,0.057027
38,team_per_poss_pf,0.043408
...,...,...
81,team_ft_fga,0.000000
82,team_efg_pct1,0.000000
83,team_tov_pct1,0.000000
84,team_drb_pct,0.000000


In [116]:
# get features to drop from df_importances
features_to_drop = df_importances[df_importances["Importance_team_score"] == 0.000]["Feature"].tolist()

# create new filtered dataframe
df_league_filtered_v6 = df_league_filtered_v5.drop(columns=features_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v6.shape}")

Dropped 44 features. New shape: (2040, 49)


In [117]:
team_name = "League"
model_df = df_league_filtered_v6

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.01071,6.690102,0.160804,8.27407,7.272575,0.054287


In [118]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.197045
36,team_sos,0.136270
21,team_per_poss_pts,0.112060
39,team_ts_pct,0.057027
20,team_per_poss_pf,0.043408
27,opp_per_poss_tov,0.035571
22,opp_per_poss_fga,0.027918
38,team_pace,0.027054
32,team_pct_of_fg_ast'd_3p,0.025910
13,team_per_poss_2p_pct,0.024969


In [119]:
# create new filtered dataframe
df_league_filtered_v7 = df_league_filtered_v6.drop(columns='team_fg_pct_by_distance_16_3p')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v7.shape}")

Dropped 44 features. New shape: (2040, 48)


In [120]:
team_name = "League"
model_df = df_league_filtered_v7

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.009668,6.724621,0.160388,8.274166,7.176392,0.054242


In [121]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.195503
35,team_sos,0.135204
21,team_per_poss_pts,0.113114
38,team_ts_pct,0.056581
20,team_per_poss_pf,0.043068
27,opp_per_poss_tov,0.035293
22,opp_per_poss_fga,0.027699
37,team_pace,0.026551
31,team_pct_of_fg_ast'd_3p,0.025707
13,team_per_poss_2p_pct,0.024774


In [122]:
# create new filtered dataframe
df_league_filtered_v8 = df_league_filtered_v7.drop(columns='team_corner_pct3pa')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v8.shape}")

Dropped 44 features. New shape: (2040, 47)


In [123]:
team_name = "League"
model_df = df_league_filtered_v8

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.00926,6.735622,0.160206,8.260218,7.05608,0.056389


In [124]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.189566
34,team_sos,0.145400
21,team_per_poss_pts,0.116637
37,team_ts_pct,0.060848
20,team_per_poss_pf,0.046316
27,opp_per_poss_tov,0.037955
36,team_pace,0.027963
31,team_pct_of_fg_ast'd_3p,0.027869
13,team_per_poss_2p_pct,0.026009
35,team_srs,0.024067


In [125]:
# create new filtered dataframe
df_league_filtered_v9 = df_league_filtered_v7.drop(columns='team_fg_pct_by_distance_10_16')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v9.shape}")

Dropped 44 features. New shape: (2040, 47)


In [126]:
team_name = "League"
model_df = df_league_filtered_v9

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.008894,6.718975,0.16048,8.267433,7.150608,0.055734


In [127]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.197253
34,team_sos,0.140671
21,team_per_poss_pts,0.117688
37,team_ts_pct,0.058869
20,team_per_poss_pf,0.044810
27,opp_per_poss_tov,0.036720
36,team_pace,0.027042
30,team_pct_of_fg_ast'd_3p,0.026746
13,team_per_poss_2p_pct,0.024986
35,team_srs,0.023285


In [128]:
# create new filtered dataframe
df_league_filtered_v10 = df_league_filtered_v9.drop(columns='team_corner_pct3pa')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v10.shape}")

Dropped 44 features. New shape: (2040, 46)


In [129]:
team_name = "League"
model_df = df_league_filtered_v10

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.00926,6.735622,0.160206,8.260218,7.05608,0.056389


In [130]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.189566
33,team_sos,0.145400
21,team_per_poss_pts,0.116637
36,team_ts_pct,0.060848
20,team_per_poss_pf,0.046316
27,opp_per_poss_tov,0.037955
35,team_pace,0.027963
30,team_pct_of_fg_ast'd_3p,0.027869
13,team_per_poss_2p_pct,0.026009
34,team_srs,0.024067


In [131]:
# create new filtered dataframe
df_league_filtered_v11 = df_league_filtered_v10.drop(columns='opp_per_poss_3p_pct')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v11.shape}")

Dropped 44 features. New shape: (2040, 45)


In [133]:
team_name = "League"
model_df = df_league_filtered_v11

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.011282,6.695702,0.159988,8.266928,7.156094,0.056172


In [134]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.196059
32,team_sos,0.146159
21,team_per_poss_pts,0.118231
35,team_ts_pct,0.061165
20,team_per_poss_pf,0.046558
26,opp_per_poss_tov,0.038153
34,team_pace,0.028106
29,team_pct_of_fg_ast'd_3p,0.027450
13,team_per_poss_2p_pct,0.026145
33,team_srs,0.024193


In [135]:
# create new filtered dataframe
df_league_filtered_v12 = df_league_filtered_v10.drop(columns='team_age')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v12.shape}")

Dropped 44 features. New shape: (2040, 44)


In [136]:
team_name = "League"
model_df = df_league_filtered_v12

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.011282,6.695702,0.159988,8.266928,7.156094,0.056172


In [137]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.196059
31,team_sos,0.146159
21,team_per_poss_pts,0.118231
34,team_ts_pct,0.061165
20,team_per_poss_pf,0.046558
26,opp_per_poss_tov,0.038153
33,team_pace,0.028106
29,team_pct_of_fg_ast'd_3p,0.027450
13,team_per_poss_2p_pct,0.026145
32,team_srs,0.024193


In [138]:
# create new filtered dataframe
df_league_filtered_v13 = df_league_filtered_v10.drop(columns='team_age')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v13.shape}")

Dropped 44 features. New shape: (2040, 45)


In [139]:
team_name = "League"
model_df = df_league_filtered_v13

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.00926,6.735622,0.160206,8.260218,7.05608,0.056389


In [140]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.189566
32,team_sos,0.145400
21,team_per_poss_pts,0.116637
35,team_ts_pct,0.060848
20,team_per_poss_pf,0.046316
27,opp_per_poss_tov,0.037955
34,team_pace,0.027963
30,team_pct_of_fg_ast'd_3p,0.027869
13,team_per_poss_2p_pct,0.026009
33,team_srs,0.024067


In [141]:
# create new filtered dataframe
df_league_filtered_v14 = df_league_filtered_v10.drop(columns='team_per_poss_ft')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v14.shape}")

Dropped 44 features. New shape: (2040, 45)


In [143]:
team_name = "League"
model_df = df_league_filtered_v14

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.005577,6.736881,0.160785,8.291044,7.089989,0.049172


In [144]:
# create new filtered dataframe
df_league_filtered_v15 = df_league_filtered_v10.drop(columns='team_per_poss_fta')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v15.shape}")

Dropped 44 features. New shape: (2040, 45)


In [145]:
team_name = "League"
model_df = df_league_filtered_v15

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.010041,6.684284,0.160108,8.250869,7.083351,0.060036


In [146]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.190043
32,team_sos,0.143694
20,team_per_poss_pts,0.111166
35,team_ts_pct,0.070065
19,team_per_poss_pf,0.045773
26,opp_per_poss_tov,0.037509
34,team_pace,0.027669
13,team_per_poss_2p_pct,0.025704
29,team_pct_of_fg_ast'd_3p,0.024848
33,team_srs,0.022040


In [147]:
# create new filtered dataframe
df_league_filtered_v16 = df_league_filtered_v15.drop(columns='team_per_poss_pf')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v16.shape}")

Dropped 44 features. New shape: (2040, 44)


In [148]:
team_name = "League"
model_df = df_league_filtered_v16

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.008354,6.701427,0.160352,8.253769,7.154694,0.059541


In [149]:
# create new filtered dataframe
df_league_filtered_v17 = df_league_filtered_v15.drop(columns='team_per_poss_ft_pct')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v17.shape}")

Dropped 44 features. New shape: (2040, 44)


In [150]:
team_name = "League"
model_df = df_league_filtered_v17

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.012418,6.705261,0.159749,8.274298,7.344814,0.055909


In [151]:
# create new filtered dataframe
df_league_filtered_v18 = df_league_filtered_v15.drop(columns='team_orb_pct')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v18.shape}")

Dropped 44 features. New shape: (2040, 44)


In [152]:
team_name = "League"
model_df = df_league_filtered_v18

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.009525,6.712044,0.160464,8.249537,7.114506,0.058431


In [153]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
9,team_per_poss_fg,0.172794
32,team_sos,0.143680
20,team_per_poss_pts,0.109419
35,team_ts_pct,0.070059
19,team_per_poss_pf,0.045768
26,opp_per_poss_tov,0.037505
29,team_pct_of_fg_ast'd_3p,0.029672
34,team_pace,0.027959
13,team_per_poss_2p_pct,0.027512
2,corner_3_specialist,0.025313


In [156]:
# clearly identify columns containing 'opp_'
cols_to_drop = [
    col for col in df_league_filtered_v15.columns
    if 'opp_' in col and col not in ['opp_score', 'opp']
]

# create new filtered dataframe
df_league_filtered_v19 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v19.shape}")

Dropped 44 features. New shape: (2040, 38)


In [157]:
team_name = "League"
model_df = df_league_filtered_v19

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.00724,6.721283,0.159728,8.269459,7.320667,0.056369


In [158]:
# create new filtered dataframe
df_league_filtered_v20 = df_league_filtered_v15.drop(columns='opp_per_poss_2p_pct')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v20.shape}")

Dropped 44 features. New shape: (2040, 44)


In [159]:
team_name = "League"
model_df = df_league_filtered_v20

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.007559,6.74963,0.160702,8.250968,7.140099,0.058269


In [160]:
# create new filtered dataframe
df_league_filtered_v21 = df_league_filtered_v15.drop(columns='slasher')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v21.shape}")

Dropped 44 features. New shape: (2040, 44)


In [161]:
team_name = "League"
model_df = df_league_filtered_v21

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.011123,6.75412,0.160252,8.253313,7.225967,0.061012


In [162]:
cols_to_drop = [
    "catch_and_shoot",
    "opp_per_poss_fga",
    "team_per_poss_ft_pct",
    "home_away",
    "opp_per_poss_fg_pct",
    "opp_per_poss_pf",
    "heave_chucker",
    "team_mov",
    "efficient_scorer",
    "plus_minus_driver",
    "opp_per_poss_2p_pct",
    "opp_per_poss_3p_pct",
    "team_age",
    "slasher"
]

# create new filtered dataframe
df_league_filtered_v22 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v22.shape}")

Dropped 44 features. New shape: (2040, 31)


In [163]:
team_name = "League"
model_df = df_league_filtered_v22

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.034943,6.595501,0.153458,8.172892,7.239738,0.079015


In [164]:
cols_to_drop = [
    "catch_and_shoot",
    "opp_per_poss_fga",
    "team_per_poss_ft_pct",
    # "home_away",
    "opp_per_poss_fg_pct",
    "opp_per_poss_pf",
    "heave_chucker",
    "team_mov",
    "efficient_scorer",
    "plus_minus_driver",
    "opp_per_poss_2p_pct",
    "opp_per_poss_3p_pct",
    "team_age",
    "slasher"
]

# create new filtered dataframe
df_league_filtered_v23 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v23.shape}")

Dropped 44 features. New shape: (2040, 32)


In [165]:
team_name = "League"
model_df = df_league_filtered_v23

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.028347,6.711761,0.157355,8.189313,7.107536,0.074567


In [166]:
cols_to_drop = [
    "catch_and_shoot",
    "opp_per_poss_fga",
    "team_per_poss_ft_pct",
    "home_away",
    "opp_per_poss_fg_pct",
    "opp_per_poss_pf",
    "heave_chucker",
    "team_mov",
    "efficient_scorer",
    "plus_minus_driver",
    "opp_per_poss_2p_pct",
    "opp_per_poss_3p_pct",
    # "team_age",
    "slasher"
]

# create new filtered dataframe
df_league_filtered_v24 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v24.shape}")

Dropped 44 features. New shape: (2040, 32)


In [167]:
team_name = "League"
model_df = df_league_filtered_v24

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.03553,6.59177,0.153264,8.173504,7.213524,0.078821


In [168]:
cols_to_drop = [
    # "catch_and_shoot",
    "opp_per_poss_fga",
    "team_per_poss_ft_pct",
    "home_away",
    "opp_per_poss_fg_pct",
    "opp_per_poss_pf",
    "heave_chucker",
    "team_mov",
    "efficient_scorer",
    "plus_minus_driver",
    "opp_per_poss_2p_pct",
    "opp_per_poss_3p_pct",
    "team_age",
    "slasher"
]

# create new filtered dataframe
df_league_filtered_v25 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v25.shape}")

Dropped 44 features. New shape: (2040, 32)


In [169]:
team_name = "League"
model_df = df_league_filtered_v25

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.032294,6.596363,0.154336,8.267672,7.386543,0.061554


In [170]:
cols_to_drop = [
    "catch_and_shoot",
    # "opp_per_poss_fga",
    "team_per_poss_ft_pct",
    "home_away",
    "opp_per_poss_fg_pct",
    "opp_per_poss_pf",
    "heave_chucker",
    "team_mov",
    "efficient_scorer",
    "plus_minus_driver",
    "opp_per_poss_2p_pct",
    "opp_per_poss_3p_pct",
    "team_age",
    "slasher"
]

# create new filtered dataframe
df_league_filtered_v26 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v26.shape}")

Dropped 44 features. New shape: (2040, 32)


In [171]:
team_name = "League"
model_df = df_league_filtered_v26

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.03612,6.622162,0.153436,8.176748,7.186958,0.078595


In [173]:
cols_to_drop = [
    "catch_and_shoot",
    # "opp_per_poss_fga",
    "team_per_poss_ft_pct",
    # "home_away",
    "opp_per_poss_fg_pct",
    "opp_per_poss_pf",
    "heave_chucker",
    "team_mov",
    # "efficient_scorer",
    "plus_minus_driver",
    "opp_per_poss_2p_pct",
    "opp_per_poss_3p_pct",
    # "team_age",
    "slasher"
]

# create new filtered dataframe
df_league_filtered_v27 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v27.shape}")

Dropped 44 features. New shape: (2040, 35)


In [174]:
team_name = "League"
model_df = df_league_filtered_v27

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.014863,6.703465,0.15894,8.259248,7.205605,0.063116


In [175]:
cols_to_drop = [
    "catch_and_shoot",
    "opp_per_poss_fga",
    "team_per_poss_ft_pct",
    "home_away",
    "opp_per_poss_fg_pct",
    "opp_per_poss_pf",
    "heave_chucker",
    "team_mov",
    # "efficient_scorer",
    "plus_minus_driver",
    "opp_per_poss_2p_pct",
    "opp_per_poss_3p_pct",
    "team_age",
    "slasher"
]

# create new filtered dataframe
df_league_filtered_v28 = df_league_filtered_v15.drop(columns=cols_to_drop)

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v28.shape}")

Dropped 44 features. New shape: (2040, 32)


In [176]:
team_name = "League"
model_df = df_league_filtered_v28

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.022494,6.5583,0.156002,8.230828,7.166759,0.067726


In [177]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
4,team_per_poss_fg,0.240214
14,team_per_poss_pts,0.158285
19,team_sos,0.130782
22,team_ts_pct,0.077860
13,team_per_poss_pf,0.071231
21,team_pace,0.033162
20,team_srs,0.032640
5,team_per_poss_fg_pct,0.027758
10,team_per_poss_orb,0.024619
17,team_dist,0.022905


In [179]:
team_name = "League"
model_df = df_league_filtered_v22

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.034943,6.595501,0.153458,8.172892,7.239738,0.079015


In [180]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
3,team_per_poss_fg,0.219103
18,team_sos,0.218270
13,team_per_poss_pts,0.171312
12,team_per_poss_pf,0.069864
21,team_ts_pct,0.056978
15,opp_per_poss_tov,0.040975
9,team_per_poss_orb,0.032666
20,team_pace,0.027652
19,team_srs,0.025526
14,opp_per_poss_stl,0.022332


In [184]:
for col in df_league_filtered_v22.columns:
    print(col)

team
year
opp
team_score
opp_score
day
month
corner_3_specialist
midrange_sniper
self_creator
team_per_poss_fg
team_per_poss_fg_pct
team_per_poss_3p_pct
team_per_poss_2pa
team_per_poss_2p_pct
team_per_poss_ft
team_per_poss_orb
team_per_poss_ast
team_per_poss_tov
team_per_poss_pf
team_per_poss_pts
opp_per_poss_stl
opp_per_poss_tov
team_dist
team_pct_of_fg_ast'd_3p
team_sos
team_srs
team_pace
team_ts_pct
team_efg_pct
team_orb_pct


In [186]:
# create new filtered dataframe
df_league_filtered_v29 = df_league_filtered_v22.drop(columns='team_per_poss_2p_pct')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v29.shape}")

Dropped 44 features. New shape: (2040, 30)


In [187]:
team_name = "League"
model_df = df_league_filtered_v29

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.034943,6.595501,0.153458,8.172892,7.239738,0.079015


In [188]:
# create new filtered dataframe
df_league_filtered_v30 = df_league_filtered_v29.drop(columns='team_per_poss_fg_pct')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v30.shape}")

Dropped 44 features. New shape: (2040, 29)


In [189]:
team_name = "League"
model_df = df_league_filtered_v30

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.034943,6.595501,0.153458,8.172892,7.239738,0.079015


In [190]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
3,team_per_poss_fg,0.219103
16,team_sos,0.218270
11,team_per_poss_pts,0.171312
10,team_per_poss_pf,0.069864
19,team_ts_pct,0.056978
13,opp_per_poss_tov,0.040975
7,team_per_poss_orb,0.032666
18,team_pace,0.027652
17,team_srs,0.025526
12,opp_per_poss_stl,0.022332


In [193]:
# create new filtered dataframe
df_league_filtered_v31 = df_league_filtered_v30.drop(columns='team_efg_pct')

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v31.shape}")

Dropped 44 features. New shape: (2040, 28)


In [194]:
team_name = "League"
model_df = df_league_filtered_v31

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.034943,6.595501,0.153458,8.172892,7.239738,0.079015


In [195]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
3,team_per_poss_fg,0.219103
16,team_sos,0.218270
11,team_per_poss_pts,0.171312
10,team_per_poss_pf,0.069864
19,team_ts_pct,0.056978
13,opp_per_poss_tov,0.040975
7,team_per_poss_orb,0.032666
18,team_pace,0.027652
17,team_srs,0.025526
12,opp_per_poss_stl,0.022332


In [197]:
# create new filtered dataframe
df_league_filtered_v32 = df_league_filtered_v31.drop(columns="team_pct_of_fg_ast'd_3p")

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v32.shape}")

Dropped 44 features. New shape: (2040, 27)


In [198]:
team_name = "League"
model_df = df_league_filtered_v32

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.033148,6.576729,0.15382,8.179139,7.083565,0.076727


In [199]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
15,team_sos,0.227608
3,team_per_poss_fg,0.216612
11,team_per_poss_pts,0.178644
10,team_per_poss_pf,0.072853
18,team_ts_pct,0.059508
17,team_pace,0.028660
7,team_per_poss_orb,0.026803
16,team_srs,0.024871
12,opp_per_poss_stl,0.021417
0,corner_3_specialist,0.020190


In [200]:
# create new filtered dataframe
df_league_filtered_v33 = df_league_filtered_v31.drop(columns="team_orb_pct")

# Quick verification
print(f"Dropped {len(features_to_drop)} features. New shape: {df_league_filtered_v33.shape}")

Dropped 44 features. New shape: (2040, 27)


In [203]:
team_name = "League"
model_df = df_league_filtered_v33

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.036641,6.600624,0.15367,8.182482,7.115788,0.077925


In [202]:
# save to CSV
df_league_filtered_v31.to_csv("df_league_filtered_v31.csv", index=False)

# download to local machine
from google.colab import files
files.download("df_league_filtered_v31.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [204]:
# load the player game logs CSV from the data folder
df_unfiltered = pd.read_csv("LHL-final-final-project/data/all_persona_and_team_data.csv")


# preview
df_unfiltered.head()

,team,year,home_away,opp,team_score,opp_score,day,month,all_around_star,and_one_machine,...,team_3par,team_ts%,team_efg%,team_tov%,team_orb%,team_ft/fga,team_efg%.1,team_tov%.1,team_drb%,team_ft/fga.1
0,ATL,2020,1.0,DAL,105,95,26.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
1,ATL,2020,0.0,LVA,70,100,29.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
2,ATL,2020,1.0,NYL,84,78,31.0,7.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
3,ATL,2020,0.0,IND,77,93,2.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231
4,ATL,2020,1.0,PHO,74,81,4.0,8.0,0.0,0.0,...,0.237,0.518,0.484,15.5,25.3,0.172,0.513,14.1,75.7,0.231


In [205]:
for col in df_unfiltered.columns:
    print(col)

team
year
home_away
opp
team_score
opp_score
day
month
all_around_star
and_one_machine
catch_and_shoot
corner_3_specialist
defensive_anchor
defensive_rebounder
efficient_scorer
elite_scorer
fast_break_threat
floor_general
free_throw_generator
glass_cleaner
heave_chucker
impact_bench
midrange_sniper
offensive_hub
offensive_rebounder
playmaker
plus_minus_driver
rim_protector
self_creator
slasher
steal_artist
stretch_big
three_point_specialist
turnover_prone
volume_shooter
team_per_game_fg
team_per_game_fga
team_per_game_fg%
team_per_game_3p
team_per_game_3pa
team_per_game_3p%
team_per_game_2p
team_per_game_2pa
team_per_game_2p%
team_per_game_ft
team_per_game_fta
team_per_game_ft%
team_per_game_orb
team_per_game_drb
team_per_game_trb
team_per_game_ast
team_per_game_stl
team_per_game_blk
team_per_game_tov
team_per_game_pf
team_per_game_pts
opp_per_game_fg
opp_per_game_fga
opp_per_game_fg%
opp_per_game_3p
opp_per_game_3pa
opp_per_game_3p%
opp_per_game_2p
opp_per_game_2pa
opp_per_game_2p%
op

In [206]:
# Columns to select
selected_columns = ["team", "opp", "year", "team_score", "opp_score", "day", "month", "team_pace"]

# Create new DataFrame
league_df_v1 = df_unfiltered[selected_columns].copy()

# Quick verification
print(f"New shape: {league_df_v1.shape}")
league_df_v1.head()

New shape: (2040, 8)


,team,opp,year,team_score,opp_score,day,month,team_pace
0,ATL,DAL,2020,105,95,26.0,7.0,80.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0


In [207]:
team_name = "League"
model_df = league_df_v1

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.36996,6.938446,0.091633,8.595137,7.103119,-0.010998


In [208]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
0,team_pace,1.0


In [209]:
# Add 'team_ortg' to existing league_df_v1
league_df_v2 = league_df_v1.copy()
league_df_v2['team_ortg'] = df_unfiltered['team_ortg']

# Quick verification
print(f"New shape: {league_df_v2.shape}")
league_df_v2.head()

New shape: (2040, 9)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1


In [210]:
team_name = "League"
model_df = league_df_v2

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.055712,6.602715,0.147438,8.162238,7.0,0.08214


In [211]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.832695
0,team_pace,0.167305


In [213]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v3 = league_df_v2.copy()
league_df_v3['team_ts_pct'] = df_unfiltered['team_ts%']

# Quick verification
print(f"New shape: {league_df_v3.shape}")
league_df_v3.head()

New shape: (2040, 10)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,team_ts_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,0.518
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.518
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,0.518
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.518
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,0.518


In [214]:
team_name = "League"
model_df = league_df_v3

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.055696,6.600742,0.147515,8.180031,7.078491,0.08049


In [215]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.500845
2,team_ts_pct,0.402206
0,team_pace,0.096949


In [220]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v4 = league_df_v2.copy()
league_df_v4['team_efg_pct'] = df_unfiltered['team_efg%.1']

# Quick verification
print(f"New shape: {league_df_v4.shape}")
league_df_v4.head()

New shape: (2040, 10)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,team_efg_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,0.513
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.513
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,0.513
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.513
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,0.513


In [218]:
team_name = "League"
model_df = league_df_v4

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.050746,6.539185,0.147567,8.16429,7.0,0.08193


In [221]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v5 = league_df_v2.copy()
league_df_v5['team_efg_pct'] = df_unfiltered['team_efg%']

# Quick verification
print(f"New shape: {league_df_v5.shape}")
league_df_v5.head()

New shape: (2040, 10)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,team_efg_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,0.484
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.484
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,0.484
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.484
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,0.484


In [222]:
team_name = "League"
model_df = league_df_v5

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.057594,6.598076,0.147501,8.196203,6.943192,0.079438


In [223]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v6 = league_df_v2.copy()
league_df_v6['team_age'] = df_unfiltered['team_age']

# Quick verification
print(f"New shape: {league_df_v6.shape}")
league_df_v6.head()

New shape: (2040, 10)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,team_age
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,25.8
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,25.8
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,25.8
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,25.8
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,25.8


In [224]:
team_name = "League"
model_df = league_df_v6

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.054807,6.599262,0.147451,8.163394,7.0,0.082063


In [225]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.825533
0,team_pace,0.165110
2,team_age,0.009357


In [226]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v7 = league_df_v2.copy()
league_df_v7['home_away'] = df_unfiltered['home_away']

# Quick verification
print(f"New shape: {league_df_v7.shape}")
league_df_v7.head()

New shape: (2040, 10)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0


In [227]:
team_name = "League"
model_df = league_df_v7

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.051639,6.625092,0.151438,8.146606,6.901051,0.085111


In [228]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.777054
0,team_pace,0.166853
2,home_away,0.056093


In [229]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v8 = league_df_v7.copy()
league_df_v8['elite_scorer'] = df_unfiltered['elite_scorer']

# Quick verification
print(f"New shape: {league_df_v8.shape}")
league_df_v8.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,elite_scorer
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,1.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,1.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,1.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,1.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,1.0


In [230]:
team_name = "League"
model_df = league_df_v8

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.052462,6.627419,0.15145,8.147054,6.901173,0.085007


In [231]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v9 = league_df_v7.copy()
league_df_v9['team_per_poss_orb'] = df_unfiltered['team_per_poss_orb']

# Quick verification
print(f"New shape: {league_df_v9.shape}")
league_df_v9.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_per_poss_orb
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,11.2
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,11.2
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,11.2
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,11.2
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,11.2


In [232]:
team_name = "League"
model_df = league_df_v9

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.036937,6.562828,0.152734,8.154575,6.8913,0.08281


In [233]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v10 = league_df_v7.copy()
league_df_v10['opp_per_poss_tov'] = df_unfiltered['opp_per_poss_tov']

# Quick verification
print(f"New shape: {league_df_v10.shape}")
league_df_v10.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_per_poss_tov
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,15.9
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,15.9
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,15.9
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,15.9
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,15.9


In [234]:
team_name = "League"
model_df = league_df_v10

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.050451,6.689278,0.1512,8.154701,6.986973,0.082034


In [235]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v11 = league_df_v7.copy()
league_df_v11['team_dist'] = df_unfiltered['team_dist.']

# Quick verification
print(f"New shape: {league_df_v11.shape}")
league_df_v11.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_dist
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,13.1
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,13.1
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,13.1
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,13.1
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,13.1


In [236]:
team_name = "League"
model_df = league_df_v11

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.052457,6.637161,0.151436,8.149488,6.906132,0.084639


In [237]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v12 = league_df_v7.copy()
league_df_v12['team_3par'] = df_unfiltered['team_3par']

# Quick verification
print(f"New shape: {league_df_v12.shape}")
league_df_v12.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_3par
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,0.237
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,0.237
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,0.237
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,0.237
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,0.237


In [238]:
team_name = "League"
model_df = league_df_v12

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.052027,6.673447,0.151565,8.151174,6.924603,0.084139


In [239]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v13 = league_df_v7.copy()
league_df_v13['three_point_specialist'] = df_unfiltered['three_point_specialist']

# Quick verification
print(f"New shape: {league_df_v13.shape}")
league_df_v13.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,three_point_specialist
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,0.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,0.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,0.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,0.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,0.0


In [240]:
team_name = "League"
model_df = league_df_v13

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.05135,6.629391,0.151524,8.150115,6.940254,0.084565


In [241]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v14 = league_df_v7.copy()
league_df_v14['all_around_star'] = df_unfiltered['all_around_star']

# Quick verification
print(f"New shape: {league_df_v14.shape}")
league_df_v14.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,all_around_star
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,0.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,0.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,0.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,0.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,0.0


In [242]:
team_name = "League"
model_df = league_df_v14

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.051639,6.625092,0.151438,8.146606,6.901051,0.085111


In [243]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v15 = league_df_v7.copy()
league_df_v15['team_per_poss_fg_pct'] = df_unfiltered['team_per_poss_fg%']

# Quick verification
print(f"New shape: {league_df_v15.shape}")
league_df_v15.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_per_poss_fg_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,0.442
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,0.442
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,0.442
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,0.442
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,0.442


In [244]:
team_name = "League"
model_df = league_df_v15

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.039326,6.755814,0.152353,8.15612,6.920567,0.083518


In [245]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v16 = league_df_v7.copy()
league_df_v16['team_per_poss_2pa'] = df_unfiltered['team_per_poss_2pa']

# Quick verification
print(f"New shape: {league_df_v16.shape}")
league_df_v16.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_per_poss_2pa
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,67.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,67.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,67.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,67.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,67.0


In [246]:
team_name = "League"
model_df = league_df_v16

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.049749,6.640839,0.151481,8.15364,6.93066,0.083456


In [247]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v17 = league_df_v7.copy()
league_df_v17['team_per_poss_3pa'] = df_unfiltered['team_per_poss_3pa']

# Quick verification
print(f"New shape: {league_df_v17.shape}")
league_df_v17.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_per_poss_3pa
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,20.9
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,20.9
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,20.9
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,20.9
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,20.9


In [250]:
team_name = "League"
model_df = league_df_v17

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.052275,6.657547,0.151553,8.151722,6.896378,0.084638


In [249]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v18 = league_df_v7.copy()
league_df_v18['team_srs'] = df_unfiltered['team_srs']

# Quick verification
print(f"New shape: {league_df_v18.shape}")
league_df_v18.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_srs
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,-6.13
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,-6.13
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,-6.13
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,-6.13
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,-6.13


In [251]:
team_name = "League"
model_df = league_df_v18

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.049464,6.602516,0.150957,8.140844,6.82732,0.084451


In [252]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v19 = league_df_v7.copy()
league_df_v19['team_sos'] = df_unfiltered['team_sos']

# Quick verification
print(f"New shape: {league_df_v19.shape}")
league_df_v19.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,team_sos
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,0.56
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,0.56
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,0.56
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,0.56
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,0.56


In [253]:
team_name = "League"
model_df = league_df_v19

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,8.047635,6.568222,0.151899,8.15,6.784073,0.085102


In [258]:
# create a mapping from team to their drtg
team_to_drtg = df_unfiltered.set_index('team')['team_drtg'].to_dict()

# map each opponent to their team_drtg
df_unfiltered['opp_drtg'] = df_unfiltered['opp'].map(team_to_drtg)

# quick verification
print(df_unfiltered[df_unfiltered["year"] == 2024][['team', 'opp', 'team_drtg', 'opp_drtg']])

     team  opp  team_drtg  opp_drtg
1560  ATL  LAS      102.5     107.7
1561  ATL  PHO      102.5     107.8
1562  ATL  DAL      102.5     114.0
1563  ATL  MIN      102.5      96.5
1564  ATL  WAS      102.5     103.4
...   ...  ...        ...       ...
2035  WAS  CHI      103.4     105.6
2036  WAS  ATL      103.4     102.5
2037  WAS  ATL      103.4     102.5
2038  WAS  NYL      103.4      97.9
2039  WAS  IND      103.4     109.5

[480 rows x 4 columns]


In [259]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v20 = league_df_v7.copy()
league_df_v20['opp_drtg'] = df_unfiltered['opp_drtg']

# Quick verification
print(f"New shape: {league_df_v20.shape}")
league_df_v20.head()

New shape: (2040, 11)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8


In [260]:
team_name = "League"
model_df = league_df_v20

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.768417,6.344093,0.200959,7.731628,6.553509,0.178984


In [261]:
# create a mapping from team to their drtg
team_to_pace = df_unfiltered.set_index('team')['team_pace'].to_dict()

# map each opponent to their team_drtg
df_unfiltered['opp_pace'] = df_unfiltered['opp'].map(team_to_pace)

# quick verification
print(df_unfiltered[df_unfiltered["year"] == 2024][['team', 'opp', 'team_pace', 'opp_pace']])

     team  opp  team_pace  opp_pace
1560  ATL  LAS       77.1      79.2
1561  ATL  PHO       77.1      78.2
1562  ATL  DAL       77.1      80.1
1563  ATL  MIN       77.1      77.7
1564  ATL  WAS       77.1      79.1
...   ...  ...        ...       ...
2035  WAS  CHI       79.1      78.2
2036  WAS  ATL       79.1      77.1
2037  WAS  ATL       79.1      77.1
2038  WAS  NYL       79.1      78.1
2039  WAS  IND       79.1      79.8

[480 rows x 4 columns]


In [263]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v21 = league_df_v20.copy()
league_df_v21['opp_pace'] = df_unfiltered['opp_pace']

# Quick verification
print(f"New shape: {league_df_v21.shape}")
league_df_v21.head()

New shape: (2040, 12)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pace
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,80.1
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,79.7
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,78.1
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,79.8
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,78.2


In [264]:
team_name = "League"
model_df = league_df_v21

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.731063,6.289745,0.206296,7.786866,6.406891,0.170295


In [265]:
# drop 'team_pace' from your league_df_v20 DataFrame to create league_df_v22
league_df_v22 = league_df_v20.drop(columns=['team_pace'])

# Quick verification
print(f"New shape: {league_df_v22.shape}")
league_df_v22.head()

New shape: (2040, 10)


,team,opp,year,team_score,opp_score,day,month,team_ortg,home_away,opp_drtg
0,ATL,DAL,2020,105,95,26.0,7.0,100.1,1.0,114.0
1,ATL,LVA,2020,70,100,29.0,7.0,100.1,0.0,101.2
2,ATL,NYL,2020,84,78,31.0,7.0,100.1,1.0,97.9
3,ATL,IND,2020,77,93,2.0,8.0,100.1,0.0,109.5
4,ATL,PHO,2020,74,81,4.0,8.0,100.1,1.0,107.8


In [266]:
team_name = "League"
model_df = league_df_v22

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.814812,6.481155,0.190837,7.788147,6.461353,0.163608


In [267]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v23 = league_df_v20.copy()
league_df_v23['opp_pct_of_fga_by_distance_3p'] = df_unfiltered['opp_% of fga by distance_3p']

# Quick verification
print(f"New shape: {league_df_v23.shape}")
league_df_v23.head()

New shape: (2040, 12)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_3p
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.326
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.326
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.326
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.326
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.326


In [268]:
team_name = "League"
model_df = league_df_v23

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.767763,6.338947,0.201399,7.73181,6.540478,0.178853


In [269]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v24 = league_df_v20.copy()
league_df_v24['opp_fg_pct_by_distance_3p'] = df_unfiltered['opp_fg% by distance_3p']

# Quick verification
print(f"New shape: {league_df_v24.shape}")
league_df_v24.head()

New shape: (2040, 12)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_fg_pct_by_distance_3p
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.348
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.348
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.348
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.348
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.348


In [270]:
team_name = "League"
model_df = league_df_v24

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.761135,6.372772,0.202784,7.73955,6.558807,0.177791


In [271]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v25 = league_df_v20.copy()
league_df_v25['opp_pct_of_fga_by_distance_0_3'] = df_unfiltered['opp_% of fga by distance_0-3']

# Quick verification
print(f"New shape: {league_df_v25.shape}")
league_df_v25.head()

New shape: (2040, 12)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121


In [272]:
team_name = "League"
model_df = league_df_v25

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.761438,6.285892,0.202288,7.722012,6.449089,0.179871


In [273]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v26 = league_df_v25.copy()
league_df_v26['team_pct_of_fga_by_distance_0_3'] = df_unfiltered['team_% of fga by distance_0-3']

# Quick verification
print(f"New shape: {league_df_v26.shape}")
league_df_v26.head()

New shape: (2040, 13)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13


In [275]:
team_name = "League"
model_df = league_df_v26

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.761623,6.28064,0.202341,7.721701,6.457729,0.180158


In [276]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v27 = league_df_v26.copy()
league_df_v27['team_pct_of_fga_by_distance_3p'] = df_unfiltered['team_% of fga by distance_3p']

# Quick verification
print(f"New shape: {league_df_v27.shape}")
league_df_v27.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_3p
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.237
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.237
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,0.237
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.237
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,0.237


In [277]:
team_name = "League"
model_df = league_df_v27

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.761806,6.274784,0.202489,7.730091,6.476097,0.179382


In [278]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v28 = league_df_v26.copy()
league_df_v28['team_per_poss_3p_pct'] = df_unfiltered['team_per_poss_3p%']

# Quick verification
print(f"New shape: {league_df_v28.shape}")
league_df_v28.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,team_per_poss_3p_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.35
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.35
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,0.35
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.35
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,0.35


In [279]:
team_name = "League"
model_df = league_df_v28

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.76443,6.327431,0.203291,7.759278,6.545208,0.171548


In [280]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v29 = league_df_v26.copy()
league_df_v29['team_per_poss_2p_pct'] = df_unfiltered['team_per_poss_2p%']

# Quick verification
print(f"New shape: {league_df_v29.shape}")
league_df_v29.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,team_per_poss_2p_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.471
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.471
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,0.471
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.471
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,0.471


In [282]:
team_name = "League"
model_df = league_df_v29

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.746321,6.294067,0.203636,7.746577,6.497173,0.174332


In [283]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v30 = league_df_v26.copy()
league_df_v30['team_per_game_3p_pct'] = df_unfiltered['team_per_game_3p%']

# Quick verification
print(f"New shape: {league_df_v30.shape}")
league_df_v30.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,team_per_game_3p_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.35
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.35
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,0.35
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.35
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,0.35


In [284]:
team_name = "League"
model_df = league_df_v30

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.76443,6.327431,0.203291,7.759278,6.545208,0.171548


In [285]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v31 = league_df_v26.copy()
league_df_v31['team_per_poss_fta'] = df_unfiltered['team_per_poss_fta']

# Quick verification
print(f"New shape: {league_df_v31.shape}")
league_df_v31.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,team_per_poss_fta
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,20.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,20.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,20.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,20.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,20.0


In [286]:
team_name = "League"
model_df = league_df_v31

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.761372,6.307934,0.202464,7.727609,6.459843,0.179152


In [287]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v32 = league_df_v26.copy()
league_df_v32['team_fg_pct_by_distance_3p'] = df_unfiltered['team_fg% by distance_3p']

# Quick verification
print(f"New shape: {league_df_v32.shape}")
league_df_v32.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,team_fg_pct_by_distance_3p
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.35
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.35
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,0.35
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.35
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,0.35


In [288]:
team_name = "League"
model_df = league_df_v32

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.76443,6.327431,0.203291,7.759278,6.545208,0.171548


In [292]:
# Create a mapping from team to their rim_protector score
team_to_rim_protector = df_unfiltered.set_index('team')['rim_protector'].to_dict()

# Map each opponent to their rim_protector score
df_unfiltered['opp_rim_protector'] = df_unfiltered['opp'].map(team_to_rim_protector)

# Quick verification (2024 only)
print(df_unfiltered[
    (df_unfiltered["year"] == 2024) &
    ((df_unfiltered["opp"] == "NYL"))
][['team', 'opp', 'rim_protector', 'opp_rim_protector']])

     team  opp  rim_protector  opp_rim_protector
1567  ATL  NYL            0.0                2.0
1574  ATL  NYL            0.0                2.0
1576  ATL  NYL            0.0                2.0
1599  ATL  NYL            0.0                2.0
1602  CHI  NYL            1.0                2.0
1607  CHI  NYL            2.0                2.0
1621  CHI  NYL            1.0                2.0
1622  CHI  NYL            1.0                2.0
1649  CON  NYL            0.0                2.0
1661  CON  NYL            0.0                2.0
1663  CON  NYL            0.0                2.0
1668  CON  NYL            0.0                2.0
1706  DAL  NYL            0.0                2.0
1707  DAL  NYL            0.0                2.0
1715  DAL  NYL            0.0                2.0
1716  DAL  NYL            0.0                2.0
1721  IND  NYL            0.0                2.0
1722  IND  NYL            0.0                2.0
1730  IND  NYL            0.0                2.0
1741  IND  NYL      

In [293]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v33 = league_df_v26.copy()
league_df_v33['opp_rim_protector'] = df_unfiltered['opp_rim_protector']

# Quick verification
print(f"New shape: {league_df_v33.shape}")
league_df_v33.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,opp_rim_protector
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,2.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,1.0


In [295]:
team_name = "League"
model_df = league_df_v33

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.744958,6.286549,0.204061,7.719451,6.425999,0.180308


In [297]:
# Create a mapping from team to their rim_protector score
team_to_defensive_anchor = df_unfiltered.set_index('team')['defensive_anchor'].to_dict()

# Map each opponent to their rim_protector score
df_unfiltered['opp_defensive_anchor'] = df_unfiltered['opp'].map(team_to_defensive_anchor)

# Quick verification (2024 only)
print(df_unfiltered[
    (df_unfiltered["year"] == 2024) &
    ((df_unfiltered["opp"] == "NYL"))
][['team', 'opp', 'defensive_anchor', 'opp_defensive_anchor']])

     team  opp  defensive_anchor  opp_defensive_anchor
1567  ATL  NYL               2.0                   0.0
1574  ATL  NYL               2.0                   0.0
1576  ATL  NYL               2.0                   0.0
1599  ATL  NYL               2.0                   0.0
1602  CHI  NYL               2.0                   0.0
1607  CHI  NYL               2.0                   0.0
1621  CHI  NYL               2.0                   0.0
1622  CHI  NYL               2.0                   0.0
1649  CON  NYL               2.0                   0.0
1661  CON  NYL               2.0                   0.0
1663  CON  NYL               2.0                   0.0
1668  CON  NYL               2.0                   0.0
1706  DAL  NYL               2.0                   0.0
1707  DAL  NYL               2.0                   0.0
1715  DAL  NYL               2.0                   0.0
1716  DAL  NYL               2.0                   0.0
1721  IND  NYL               2.0                   0.0
1722  IND 

In [298]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v34 = league_df_v33.copy()
league_df_v34['opp_defensive_anchor'] = df_unfiltered['opp_defensive_anchor']

# Quick verification
print(f"New shape: {league_df_v34.shape}")
league_df_v34.head()

New shape: (2040, 15)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,opp_rim_protector,opp_defensive_anchor
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.0,0.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.0,0.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,2.0,0.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.0,0.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,1.0,1.0


In [299]:
team_name = "League"
model_df = league_df_v34

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.723962,6.295311,0.207142,7.742178,6.496929,0.176323


In [300]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v35 = league_df_v33.copy()
league_df_v35['floor_general'] = df_unfiltered['floor_general']

# Quick verification
print(f"New shape: {league_df_v35.shape}")
league_df_v35.head()

New shape: (2040, 15)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,team_pct_of_fga_by_distance_0_3,opp_rim_protector,floor_general
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.13,0.0,1.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.13,0.0,1.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,0.13,2.0,1.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.13,0.0,1.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,0.13,1.0,1.0


In [301]:
team_name = "League"
model_df = league_df_v35

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.738495,6.351948,0.204483,7.730274,6.452072,0.179092


In [302]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.450407
3,opp_drtg,0.174573
0,team_pace,0.146577
4,opp_pct_of_fga_by_distance_0_3,0.087003
7,floor_general,0.054222
2,home_away,0.049508
6,opp_rim_protector,0.037710
5,team_pct_of_fga_by_distance_0_3,0.000000


In [304]:
# drop 'team_pace' from your league_df_v20 DataFrame to create league_df_v22
league_df_v36 = league_df_v35.drop(columns=['team_pct_of_fga_by_distance_0_3'])

# Quick verification
print(f"New shape: {league_df_v36.shape}")
league_df_v36.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,opp_rim_protector,floor_general
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.0,1.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.0,1.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,2.0,1.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.0,1.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,1.0,1.0


In [305]:
team_name = "League"
model_df = league_df_v36

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.738495,6.351948,0.204483,7.730274,6.452072,0.179092


In [306]:
team_name = "League"
model_df = league_df_v33

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.744958,6.286549,0.204061,7.719451,6.425999,0.180308


In [307]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.478483
3,opp_drtg,0.181608
0,team_pace,0.152925
4,opp_pct_of_fga_by_distance_0_3,0.093420
2,home_away,0.053406
6,opp_rim_protector,0.040158
5,team_pct_of_fga_by_distance_0_3,0.000000


In [308]:
# drop 'team_pace' from your league_df_v20 DataFrame to create league_df_v22
league_df_v37 = league_df_v33.drop(columns=['team_pct_of_fga_by_distance_0_3'])

# Quick verification
print(f"New shape: {league_df_v37.shape}")
league_df_v37.head()

New shape: (2040, 13)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,opp_rim_protector
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,2.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,1.0


In [309]:
team_name = "League"
model_df = league_df_v37

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.744958,6.286549,0.204061,7.719451,6.425999,0.180308


In [310]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.478483
3,opp_drtg,0.181608
0,team_pace,0.152925
4,opp_pct_of_fga_by_distance_0_3,0.093420
2,home_away,0.053406
5,opp_rim_protector,0.040158


In [ ]:
League_xgb_params = {
    'n_estimators': 300,
    'max_depth': 2,
    'learning_rate': 0.02,
    'subsample': 1,
    'colsample_bytree': 1,
    'min_child_weight': 1
}

In [311]:
# Define parameter grid for tuning
param_grid = {
    'n_estimators': [200, 300, 400],
    'max_depth': [1, 2, 3],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.9, 1],
    'colsample_bytree': [0.9, 1.0],
    'min_child_weight': [1, 2]
}

# define model with base parameters
base_model = XGBRegressor(random_state=42, verbosity=0)

# set up GridSearchCV
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=1,
    n_jobs=-1
)

# fit grid search
grid_search.fit(X_train, y_train)

# print best parameters
print("Best params:", grid_search.best_params_)

# define model with best params and retrain
best_model = grid_search.best_estimator_

# predictions using best model
y_pred_test = best_model.predict(X_test)
y_pred_train = best_model.predict(X_train)

# metrics
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": np.abs(y_pred_train - y_train).mean(),
    "Train_MAE_median": np.median(np.abs(y_pred_train - y_train)),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": np.abs(y_pred_test - y_test).mean(),
    "Test_MAE_median": np.median(np.abs(y_pred_test - y_test)),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 1, 'min_child_weight': 1, 'n_estimators': 300, 'subsample': 0.9}


,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.81378,6.350033,0.192986,7.740765,6.354557,0.177954


In [313]:
team_name = "League"
model_df = league_df_v37

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.744958,6.286549,0.204061,7.719451,6.425999,0.180308


In [315]:
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold

# Define your model
model = XGBRegressor(**team_params, random_state=42, verbosity=0)

# Set up 10-fold CV
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Run CV predictions
cv_predictions = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=-1)

# Calculate row-level MAE from CV predictions
cv_mae = np.abs(cv_predictions - y_train)

# CV Metrics
print(f"CV MAE Min: {cv_mae.min():.4f}")
print(f"CV MAE Max: {cv_mae.max():.4f}")
print(f"CV MAE Mean: {cv_mae.mean():.4f}")
print(f"CV MAE Median: {np.median(cv_mae):.4f}")
print(f"CV R²: {r2_score(y_train, cv_predictions):.4f}")

CV MAE Min: 0.0134
CV MAE Max: 36.9421
CV MAE Mean: 8.0035
CV MAE Median: 6.4914
CV R²: 0.1503


In [316]:
cv_predictions

array([81.97133 , 81.113556, 81.52998 , ..., 91.27904 , 92.88025 ,
       92.87687 ], dtype=float32)

In [319]:
# Create CV predictions DataFrame with team and opp included
cv_results_df = pd.DataFrame({
    'team': meta_train['team'],
    'opp': meta_train['opp'],
    'actual_team_score': y_train.values,
    'predicted_team_score_cv': cv_predictions,
    'mae_cv': np.abs(cv_predictions - y_train.values)
})

# Quick verification
print(cv_results_df)

     team  opp  actual_team_score  predicted_team_score_cv     mae_cv
0     ATL  DAL                105                81.971329  23.028671
1     ATL  LVA                 70                81.113556  11.113556
2     ATL  NYL                 84                81.529984   2.470016
3     ATL  IND                 77                84.238480   7.238480
4     ATL  PHO                 74                81.121307   7.121307
...   ...  ...                ...                      ...        ...
1555  LVA  NYL                 85                91.158836   6.158836
1556  LVA  WAS                 84                90.902611   6.902611
1557  LVA  SEA                103                91.279037  11.720963
1558  LVA  PHO                 94                92.880249   1.119751
1559  LVA  PHO                100                92.876869   7.123131

[1560 rows x 5 columns]


In [322]:
# Calculate average actual team_score per team (train data only)
team_avg_scores = train_df.groupby('team')['team_score'].mean().to_dict()

# Map average scores into your cv_results_df
cv_results_df['team_avg_score'] = cv_results_df['team'].map(team_avg_scores)

# Quick verification
print(cv_results_df)

     team  opp  actual_team_score  predicted_team_score_cv     mae_cv  \
0     ATL  DAL                105                81.971329  23.028671   
1     ATL  LVA                 70                81.113556  11.113556   
2     ATL  NYL                 84                81.529984   2.470016   
3     ATL  IND                 77                84.238480   7.238480   
4     ATL  PHO                 74                81.121307   7.121307   
...   ...  ...                ...                      ...        ...   
1555  LVA  NYL                 85                91.158836   6.158836   
1556  LVA  WAS                 84                90.902611   6.902611   
1557  LVA  SEA                103                91.279037  11.720963   
1558  LVA  PHO                 94                92.880249   1.119751   
1559  LVA  PHO                100                92.876869   7.123131   

      team_avg_score  
0          80.200000  
1          80.200000  
2          80.200000  
3          80.200000  
4       

In [323]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v38 = league_df_v37.copy()
league_df_v38['team_per_poss_pts'] = df_unfiltered['team_per_poss_pts']

# Quick verification
print(f"New shape: {league_df_v38.shape}")
league_df_v38.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,opp_rim_protector,team_per_poss_pts
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.0,100.1
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.0,100.1
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,2.0,100.1
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.0,100.1
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,1.0,100.1


In [324]:
team_name = "League"
model_df = league_df_v38

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.744958,6.286549,0.204061,7.719451,6.425999,0.180308


In [325]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v39 = league_df_v37.copy()
league_df_v39['team_per_game_pts'] = df_unfiltered['team_per_game_pts']

# Quick verification
print(f"New shape: {league_df_v39.shape}")
league_df_v39.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,opp_rim_protector,team_per_game_pts
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.0,81.0
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.0,81.0
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,2.0,81.0
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.0,81.0
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,1.0,81.0


In [327]:
team_name = "League"
model_df = league_df_v39

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.726803,6.262947,0.207315,7.742151,6.434856,0.17688


In [328]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v40 = league_df_v37.copy()
league_df_v40['opp_per_game_pf'] = df_unfiltered['opp_per_game_pf']

# Quick verification
print(f"New shape: {league_df_v40.shape}")
league_df_v40.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,opp_rim_protector,opp_per_game_pf
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.0,16.7
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.0,16.7
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,2.0,16.7
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.0,16.7
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,1.0,16.7


In [329]:
team_name = "League"
model_df = league_df_v40

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.743012,6.291111,0.204586,7.706477,6.474236,0.179742


In [330]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v41 = league_df_v37.copy()
league_df_v41['team_tov_pct'] = df_unfiltered['team_tov%']

# Quick verification
print(f"New shape: {league_df_v41.shape}")
league_df_v41.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,opp_rim_protector,team_tov_pct
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.0,15.5
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.0,15.5
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,2.0,15.5
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.0,15.5
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,1.0,15.5


In [331]:
team_name = "League"
model_df = league_df_v41

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.742721,6.299835,0.205384,7.73133,6.435635,0.177285


In [332]:
# Create league_df_v3 by adding 'team_ts_pct'
league_df_v42 = league_df_v37.copy()
league_df_v42['team_ftr'] = df_unfiltered['team_ftr']

# Quick verification
print(f"New shape: {league_df_v42.shape}")
league_df_v42.head()

New shape: (2040, 14)


,team,opp,year,team_score,opp_score,day,month,team_pace,team_ortg,home_away,opp_drtg,opp_pct_of_fga_by_distance_0_3,opp_rim_protector,team_ftr
0,ATL,DAL,2020,105,95,26.0,7.0,80.0,100.1,1.0,114.0,0.121,0.0,0.227
1,ATL,LVA,2020,70,100,29.0,7.0,80.0,100.1,0.0,101.2,0.121,0.0,0.227
2,ATL,NYL,2020,84,78,31.0,7.0,80.0,100.1,1.0,97.9,0.121,2.0,0.227
3,ATL,IND,2020,77,93,2.0,8.0,80.0,100.1,0.0,109.5,0.121,0.0,0.227
4,ATL,PHO,2020,74,81,4.0,8.0,80.0,100.1,1.0,107.8,0.121,1.0,0.227


In [333]:
team_name = "League"
model_df = league_df_v42

train_df = model_df[model_df["year"] < 2024]
test_df = model_df[model_df["year"] == 2024]

X_train = train_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_train = train_df["team_score"]

X_test = test_df.drop(columns=["team", "opp", "team_score", "opp_score", "day", "month", "year"])
y_test = test_df["team_score"]

# meta info for tracking predictions
meta_test = test_df[["team", "opp", "team_score"]].reset_index(drop=True)
meta_train = train_df[["team", "opp", "team_score"]].reset_index(drop=True)

# pull params from your saved dictionary
team_params = all_team_xgb_team_score_params[team_name]

# define and train model with dynamic team params
model = XGBRegressor(**team_params, random_state=42, verbosity=0)
model.fit(X_train, y_train)

# make predictions
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

# calculate row-level MAE
team_score_mae_test = np.abs(y_pred_test - y_test.values)
team_score_mae_train = np.abs(y_pred_train - y_train.values)

# store prediction rows (test)
predictions_test_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_test["team"],
    "opp": meta_test["opp"],
    "team_score": meta_test["team_score"],
    "team_score_pred": y_pred_test,
    "team_score_mae": team_score_mae_test
})

# store prediction rows (train)
predictions_train_df = pd.DataFrame({
    "Model": team_name,
    "team": meta_train["team"],
    "opp": meta_train["opp"],
    "team_score": meta_train["team_score"],
    "team_score_pred": y_pred_train,
    "team_score_mae": team_score_mae_train
})

# summary metrics (train and test)
results_df = pd.DataFrame([{
    "Model": team_name,
    "Train_MAE_mean": team_score_mae_train.mean(),
    "Train_MAE_median": np.median(team_score_mae_train),
    "Train_R2": r2_score(y_train, y_pred_train),
    "Test_MAE_mean": team_score_mae_test.mean(),
    "Test_MAE_median": np.median(team_score_mae_test),
    "Test_R2": r2_score(y_test, y_pred_test)
}])

# display summary metrics
results_df

,Model,Train_MAE_mean,Train_MAE_median,Train_R2,Test_MAE_mean,Test_MAE_median,Test_R2
0,League,7.74265,6.321621,0.204522,7.716304,6.45929,0.180956


In [334]:
# get feature names (must match what was used to train)
feature_names = X_train.columns.tolist()

# extract feature importances from the single XGBRegressor
importances_team = model.feature_importances_

df_importances = pd.DataFrame({
    "Feature": feature_names,
    "Importance_team_score": importances_team
}).sort_values("Importance_team_score", ascending=False)

df_importances

,Feature,Importance_team_score
1,team_ortg,0.454360
3,opp_drtg,0.177474
0,team_pace,0.146945
4,opp_pct_of_fga_by_distance_0_3,0.088532
2,home_away,0.051227
6,team_ftr,0.043529
5,opp_rim_protector,0.037933
